# KOH-activated carbon — surface-area & yield research notebook

**Repository:** [VishwajithJS/KoH-Surface-Area-Predictor](https://github.com/VishwajithJS/KoH-Surface-Area-Predictor)

This notebook runs the whole research and figure pipeline in **Google Colab** (`Runtime ▸ Run all`).
It uses only the measured laboratory data in `data/AC_Prep_-_KOH.txt`:

| Quantity | Symbol in data | Levels / range | Unit |
|---|---|---|---|
| Impregnation ratio (KOH : carbon) | `IR` | 0, 1, 2, 3, 4, 5, 6 | – (mass ratio) |
| Activation temperature | `Act_Temp` | 800, 900, 1000, 1100, 1200 | °C |
| Activation time | `Act_Time` | 0, 15, 30, 45, 60, 75, 90 | min |
| Surface area (target) | `Sur_Area` | measured for all 245 grid points | see `SA_UNIT` in the configuration |
| Yield (second target) | `Yield` | measured for IR = 4 (35 points) + 1 point each at IR 1, 2, 3, 5, 6 | % |

**Two parts**
* **Part A — reference-paper analyses.** The figure types of Hassan & Kazemi, *Sci. Rep.* 15, 15157 (2025)
  (scatter matrix, correlation heatmap, Monte Carlo outlier analysis, Taylor diagrams, actual-vs-predicted, cross plots,
  relative deviation, prediction distributions, Random-Forest and SHAP importance) recreated with **our** data and the same
  15 model families, using an independent train / validation / test split.
* **Part B — project-specific analyses.** Leave-one-temperature-out and leave-one-time-out evaluation, Gaussian-process
  uncertainty, prediction-interval calibration, response surfaces and active-learning experiment design.

**Scientific-integrity rules applied throughout:** only measured observations are used (nothing is imputed or invented);
every metric is computed in this run from held-out predictions; hyper-parameters are tuned inside training data only;
sample counts, units and the validation method are printed on figures and tables; analyses that the data cannot support
are listed explicitly in the final summary.

All outputs are written to Google Drive: `MyDrive/KoH_Surface_Area_Research/`.

## 0. Configuration
Change settings here, then use `Runtime ▸ Run all`. To analyse **yield** instead of surface area, set `TARGET = "Yield"`
and run all again (outputs are stored side by side, prefixed with the target name).

In [ ]:
# ---------------- Data & repository ----------------
PROJECT_NAME    = "KoH_Surface_Area_Research"      # folder created inside MyDrive
REPO_URL        = "https://github.com/VishwajithJS/KoH-Surface-Area-Predictor.git"
REPO_BRANCH     = "main"
REPO_NAME       = "KoH-Surface-Area-Predictor"
DATA_FILE_RAW   = "AC_Prep_-_KOH.txt"              # original lab file = source of truth (never modified)
DATA_FILE_CSV   = "AC_Prep_KOH_clean.csv"          # cleaned copy, used only for a consistency check

# ---------------- Experiment ----------------
TARGET          = "Sur_Area"     # "Sur_Area" or "Yield"  (Yield uses the IR = 4 rows only - the only IR with a full grid)
SEED            = 42
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)   # train / validation / test (Part A)
INNER_CV_FOLDS  = 5                    # folds for hyper-parameter tuning INSIDE the training data
REMOVE_OUTLIERS = False                # True = drop Monte-Carlo-flagged rows before modelling (not recommended: no replicates)
MCOD_ITERATIONS = 500                  # Monte Carlo sub-models for the outlier analysis
RUN_GROUPED_BENCHMARK = True           # leave-one-temperature-out / leave-one-time-out for all 15 models (~5-10 min on CPU)
GROUP_TEMPS     = (900, 1000, 1100)    # interior temperatures held out one at a time (800/1200 would be extrapolation)
GROUP_TIMES     = (15, 30, 45, 60, 75) # interior times held out one at a time
FORCE_RETRAIN   = False                # True = ignore models/predictions cached in Drive and retrain everything

# ---------------- Hardware ----------------
USE_GPU         = "auto"   # "auto", True or False. Used for the neural networks (ANN, CNN).
GPU_FOR_TREES   = False    # XGBoost / CatBoost on GPU are SLOWER than CPU for a few hundred rows; enable only for big data
NN_EPOCHS       = 1500
GP_RESTARTS     = 8

# ---------------- Units (shown on every axis) ----------------
SA_UNIT         = "m²/g"   # ASSUMED: BET specific surface area. Confirm against the lab method before publishing.
YIELD_UNIT      = "%"

# ---------------- Figures ----------------
FIG_FORMATS     = ("png", "pdf", "svg")
FIG_DPI         = 300
SHOW_FIGURES    = True
CONFIG_VERSION  = "1.0"

## 1. Environment setup and dependency installation
Installs only what is missing (in Colab normally just `catboost`), prints runtime information, package versions and the
GPU status, fixes all random seeds and defines the shared helper functions.

In [ ]:
import sys, os, re, subprocess, importlib, importlib.util, platform, time, json, shutil, hashlib, random, traceback, warnings, datetime
from pathlib import Path
T_START = time.time()

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn", "matplotlib": "matplotlib",
            "seaborn": "seaborn", "joblib": "joblib", "tqdm": "tqdm", "xgboost": "xgboost>=2.0", "lightgbm": "lightgbm",
            "catboost": "catboost", "shap": "shap", "torch": "torch"}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing missing packages:", " ".join(_missing))
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing])
    if _r.returncode != 0:
        raise RuntimeError("pip install failed - see the messages above.")
    importlib.invalidate_caches()
else:
    print("All required packages are already installed.")

In [ ]:
import numpy as np, pandas as pd, scipy, sklearn, matplotlib, seaborn as sns, joblib, xgboost, lightgbm, catboost, shap, torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold, GroupKFold, train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# ---- runtime information ----
print(f"Run started      : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Environment      : {'Google Colab' if IN_COLAB else 'NOT Colab (local / other Jupyter)'}")
print(f"Python           : {sys.version.split()[0]}  on {platform.platform()}")
print(f"CPU cores        : {os.cpu_count()}")
try:
    import psutil; print(f"RAM              : {psutil.virtual_memory().total / 1e9:.1f} GB")
except Exception:
    pass
VERSIONS = {m.__name__: getattr(m, "__version__", "?") for m in (np, pd, scipy, sklearn, matplotlib, sns, joblib, xgboost, lightgbm, catboost, shap, torch)}
display(pd.DataFrame(VERSIONS.items(), columns=["package", "version"]).set_index("package").T)

# ---- GPU detection (never assumed) ----
CUDA_AVAILABLE = torch.cuda.is_available()
if USE_GPU == "auto":
    USE_CUDA = CUDA_AVAILABLE
else:
    USE_CUDA = bool(USE_GPU) and CUDA_AVAILABLE
    if USE_GPU and not CUDA_AVAILABLE:
        print("WARNING: USE_GPU=True but no CUDA GPU is available - falling back to CPU.")
TORCH_DEVICE = "cuda" if USE_CUDA else "cpu"
XGB_DEVICE   = "cuda" if (USE_CUDA and GPU_FOR_TREES) else "cpu"
CAT_TASK     = "GPU" if (USE_CUDA and GPU_FOR_TREES) else "CPU"
if CUDA_AVAILABLE:
    p = torch.cuda.get_device_properties(0)
    print(f"GPU              : {p.name}, {p.total_memory / 1e9:.1f} GB  (CUDA {torch.version.cuda})")
else:
    print("GPU              : none detected (Runtime > Change runtime type > GPU to enable). CPU fallback is used.")
print(f"Device use       : neural networks -> {TORCH_DEVICE};  XGBoost -> {XGB_DEVICE};  CatBoost -> {CAT_TASK};  "
      f"other models -> CPU")

# ---- reproducibility ----
def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seeds()
N_JOBS = -1

### 1b. Shared helper functions
Error handling (`step`), figure export (`save_fig` - PNG 300 dpi + vector PDF/SVG + manifest entry), metrics and the
model builders used everywhere below.

In [ ]:
STATUS, NOTES, MANIFEST = {}, [], []
CURRENT_SECTION = "0"

class step:
    """Run a notebook section; on error print the traceback, record it and continue so that 'Run all' finishes."""
    def __init__(self, name): self.name = name
    def __enter__(self):
        global CURRENT_SECTION
        CURRENT_SECTION = self.name; self.t = time.time(); print(f"▶ {self.name}"); return self
    def __exit__(self, et, ev, tb):
        dt = time.time() - self.t
        if et is None:
            STATUS[self.name] = ("OK", round(dt, 1), ""); print(f"✔ {self.name} finished in {dt:.1f} s")
            return False
        if not issubclass(et, Exception):
            return False                     # let KeyboardInterrupt etc. through
        STATUS[self.name] = ("FAILED", round(dt, 1), f"{et.__name__}: {ev}")
        print(f"✖ {self.name} FAILED after {dt:.1f} s -> {et.__name__}: {ev}")
        traceback.print_exception(et, ev, tb)
        print("The notebook continues; sections that depend on this one may also report errors.")
        return True

def note(msg):
    """Record a caveat / unavailable analysis for the final summary."""
    NOTES.append(f"[{CURRENT_SECTION}] {msg}"); print("NOTE:", msg)

# ---- plotting style ----
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": FIG_DPI, "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "legend.fontsize": 7.5, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5, "lines.linewidth": 1.3,
    "svg.fonttype": "none", "pdf.fonttype": 42, "ps.fonttype": 42, "figure.facecolor": "white"})
SET_COLORS = {"Train": "#2b6cb0", "Validation": "#dd8a0f", "Test": "#2f855a"}
MODEL_NAMES = ["Linear Regression", "Ridge Regression", "Lasso Regression", "Elastic Net", "SVR", "Random Forest",
               "Gradient Boosting", "KNN", "Decision Tree", "XGBoost", "LightGBM", "CatBoost", "Gaussian Process", "ANN", "CNN"]
TREE_MODELS = {"Random Forest", "Gradient Boosting", "Decision Tree", "XGBoost", "LightGBM", "CatBoost"}
_MARKERS = ["o", "s", "^", "v", "D", "P", "X", "*", "<", ">", "h", "p", "8", "d", "H"]
MODEL_STYLE = {m: (plt.get_cmap("tab20")(i), _MARKERS[i]) for i, m in enumerate(MODEL_NAMES)}

def label(col):
    return {"IR": "Impregnation ratio, IR (–)", "Act_Temp": "Activation temperature (°C)", "Act_Time": "Activation time (min)",
            "Sur_Area": f"Surface area ({SA_UNIT})", "Yield": f"Yield ({YIELD_UNIT})"}.get(col, col)

def short(col):
    return {"IR": "IR", "Act_Temp": "Temperature", "Act_Time": "Time", "Sur_Area": "Surface area", "Yield": "Yield"}.get(col, col)

def save_fig(fig, key, folder, title, description, paper_ref="—", n=None, validation="—"):
    """Save a figure as PNG (FIG_DPI) + vector PDF/SVG in Drive and register it in the figure manifest."""
    fid = f"{TARGET}_{key}"
    files = []
    for ext in FIG_FORMATS:
        path = FIG_DIRS[folder] / f"{fid}.{ext}"
        try:
            fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
            files.append(str(path.relative_to(BASE)))
        except Exception as e:
            print(f"WARNING: could not save {path.name}: {e}")
    MANIFEST[:] = [m for m in MANIFEST if m["figure_id"] != fid]
    MANIFEST.append(dict(figure_id=fid, section=CURRENT_SECTION, folder=f"figures/{folder}", title=title,
                         description=description, paper_analogue=paper_ref, n=n, validation=validation, files="; ".join(files)))
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)

def save_table(df, name, index=True):
    p = DIRS["results"] / f"{TARGET}_{name}.csv"; df.to_csv(p, index=index); return p

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); e = y - p
    ss_tot = np.sum((y - y.mean()) ** 2)
    nz = y != 0
    return dict(n=len(y), R2=1 - np.sum(e ** 2) / ss_tot if ss_tot > 0 else np.nan, MSE=np.mean(e ** 2),
                RMSE=np.sqrt(np.mean(e ** 2)), MAE=np.mean(np.abs(e)), SD_res=np.std(e),
                MRD=100 * np.mean(np.abs(e[nz] / y[nz])))

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

def model_grid(n, ncols=3, w=4.1, h=3.0):
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False)
    for ax in axs.ravel()[n:]:
        ax.axis("off")
    return fig, axs.ravel()

def finish_grid(fig, title, bottom=0.0):
    """Lay out a multi-panel figure and put the overall title above it without overlapping panel titles."""
    fig.tight_layout(rect=(0, bottom, 1, 1))
    fig.suptitle(title, fontsize=9.5, y=1.0 + 0.25 / fig.get_size_inches()[1], va="bottom")

# ---- Gaussian-process builders (project model) ----
def make_gp_estimator(n_feat, restarts=GP_RESTARTS, seed=SEED):
    k = C(1.0, (1e-2, 1e2)) * Matern(length_scale=np.ones(n_feat), length_scale_bounds=(1e-2, 1e2), nu=2.5) \
        + WhiteKernel(1e-2, (1e-6, 1e0))
    return GaussianProcessRegressor(k, normalize_y=True, n_restarts_optimizer=restarts, random_state=seed)

def make_gp_pipeline(n_feat, restarts=GP_RESTARTS):
    return make_pipeline(StandardScaler(), make_gp_estimator(n_feat, restarts))

class TrendGP(BaseEstimator, RegressorMixin):
    """Linear trend + Gaussian-process residual (project yield model, see models.py). predict(..., return_std=True)."""
    def __init__(self, n_feat=2, deg=1, restarts=GP_RESTARTS):
        self.n_feat, self.deg, self.restarts = n_feat, deg, restarts
    def fit(self, X, y):
        self.sc_ = StandardScaler().fit(X); Z = self.sc_.transform(X)
        self.trend_ = make_pipeline(PolynomialFeatures(self.deg), RidgeCV(alphas=np.logspace(-4, 2, 20))).fit(Z, y)
        self.gp_ = make_gp_estimator(self.n_feat, self.restarts).fit(Z, y - self.trend_.predict(Z)); return self
    def predict(self, X, return_std=False):
        Z = self.sc_.transform(X); mu, sd = self.gp_.predict(Z, return_std=True); mu = mu + self.trend_.predict(Z)
        return (mu, sd) if return_std else mu

def gp_parts(m):
    """(scaler, fitted GaussianProcessRegressor, trend or None) of a project GP model."""
    if isinstance(m, TrendGP):
        return m.sc_, m.gp_, m.trend_
    return m[0], m[-1], None

# ---- PyTorch ANN / CNN with a scikit-learn interface (GPU when available) ----
class TorchRegressor(BaseEstimator, RegressorMixin):
    """ANN (128-64-1, ReLU) or 1-D CNN (Conv16-Conv32-Dense64-1, ReLU) as in Table 1 of the reference paper.
    Inputs and target are standardised internally (statistics from the training data only)."""
    def __init__(self, arch="ann", epochs=1500, lr=3e-3, weight_decay=1e-4, seed=42, device="cpu"):
        self.arch, self.epochs, self.lr, self.weight_decay, self.seed, self.device = arch, epochs, lr, weight_decay, seed, device
    def _build(self, n_feat):
        nn = torch.nn
        if self.arch == "ann":
            return nn.Sequential(nn.Linear(n_feat, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
        return nn.Sequential(nn.Unflatten(1, (1, n_feat)), nn.Conv1d(1, 16, 2, padding=1), nn.ReLU(),
                             nn.Conv1d(16, 32, 2, padding=1), nn.ReLU(), nn.Flatten(),
                             nn.Linear(32 * (n_feat + 2), 64), nn.ReLU(), nn.Linear(64, 1))
    def fit(self, X, y):
        X = np.asarray(X, np.float32); y = np.asarray(y, np.float32).ravel()
        self.xm_, self.xs_ = X.mean(0), X.std(0) + 1e-8
        self.ym_, self.ys_ = float(y.mean()), float(y.std() + 1e-8)
        torch.manual_seed(self.seed)
        dev = torch.device(self.device if (self.device == "cpu" or torch.cuda.is_available()) else "cpu")
        net = self._build(X.shape[1]).to(dev)
        Xt = torch.tensor((X - self.xm_) / self.xs_, device=dev)
        yt = torch.tensor((y - self.ym_) / self.ys_, device=dev).view(-1, 1)
        opt = torch.optim.Adam(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        lossf = torch.nn.MSELoss()
        for _ in range(self.epochs):
            opt.zero_grad(); loss = lossf(net(Xt), yt); loss.backward(); opt.step(); sch.step()
        self.final_loss_ = float(loss.detach().cpu()); self.net_ = net.to("cpu").eval(); return self
    def predict(self, X):
        X = np.asarray(X, np.float32)
        with torch.no_grad():
            out = self.net_(torch.tensor((X - self.xm_) / self.xs_)).numpy().ravel()
        return out * self.ys_ + self.ym_

# ---- the 15 model families of the reference paper, with search spaces modelled on its Table 1 ----
SEARCH_SPACE = {
    "Linear Regression": "none",
    "Ridge Regression": "alpha ∈ 10^[-4, 1] (11 values)",
    "Lasso Regression": "alpha ∈ 10^[-4, 1] (11 values)",
    "Elastic Net": "alpha ∈ 10^[-4, 1] (11) × l1_ratio ∈ {0.1, 0.5, 0.9}",
    "SVR": "C ∈ {1, 3, 10, 30, 100} × gamma ∈ {scale, 0.1, 0.3, 1} × epsilon ∈ {0.01, 0.05, 0.1}",
    "Random Forest": "max_depth ∈ {None, 4, 8, 16} × min_samples_leaf ∈ {1, 2}; 300 trees",
    "Gradient Boosting": "max_depth ∈ {2, 3, 6} × learning_rate ∈ {0.03, 0.1}; 300 trees",
    "KNN": "k ∈ {1, 2, 3, 4, 5, 6, 8, 10}, distance weighting",
    "Decision Tree": "max_depth ∈ {2, 3, 4, 6, 8, 12, None} × min_samples_leaf ∈ {1, 2, 4}",
    "XGBoost": "max_depth ∈ {2, 3, 6} × learning_rate ∈ {0.03, 0.1}; 400 trees",
    "LightGBM": "num_leaves ∈ {4, 8, 16} × learning_rate ∈ {0.03, 0.1}; 400 trees",
    "CatBoost": "depth ∈ {3, 5} × learning_rate ∈ {0.03, 0.1}; 400 iterations",
    "Gaussian Process": "Matérn-5/2 ARD + white noise; hyper-parameters by marginal likelihood",
    "ANN": "fixed: 128-64-1, ReLU, Adam, weight decay 1e-4",
    "CNN": "fixed: Conv1d(16)-Conv1d(32)-Dense(64)-1, ReLU, Adam, weight decay 1e-4",
}

def build_model(name, n_feat, cv):
    sc = StandardScaler
    ttr = lambda m: TransformedTargetRegressor(regressor=m, transformer=StandardScaler())
    gs = lambda est, grid: GridSearchCV(est, grid, cv=cv, scoring="neg_mean_squared_error", n_jobs=N_JOBS)
    al = np.logspace(-4, 1, 11)
    if name == "Linear Regression": return ttr(make_pipeline(sc(), LinearRegression()))
    if name == "Ridge Regression":  return gs(ttr(make_pipeline(sc(), Ridge())), {"regressor__ridge__alpha": al})
    if name == "Lasso Regression":  return gs(ttr(make_pipeline(sc(), Lasso(max_iter=50000))), {"regressor__lasso__alpha": al})
    if name == "Elastic Net":       return gs(ttr(make_pipeline(sc(), ElasticNet(max_iter=50000))),
                                              {"regressor__elasticnet__alpha": al, "regressor__elasticnet__l1_ratio": [0.1, 0.5, 0.9]})
    if name == "SVR":               return gs(ttr(make_pipeline(sc(), SVR())), {"regressor__svr__C": [1, 3, 10, 30, 100],
                                              "regressor__svr__gamma": ["scale", 0.1, 0.3, 1.0], "regressor__svr__epsilon": [0.01, 0.05, 0.1]})
    if name == "Random Forest":     return gs(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
                                              {"max_depth": [None, 4, 8, 16], "min_samples_leaf": [1, 2]})
    if name == "Gradient Boosting": return gs(GradientBoostingRegressor(n_estimators=300, random_state=SEED),
                                              {"max_depth": [2, 3, 6], "learning_rate": [0.03, 0.1]})
    if name == "KNN":               return gs(ttr(make_pipeline(sc(), KNeighborsRegressor(weights="distance"))),
                                              {"regressor__kneighborsregressor__n_neighbors": [1, 2, 3, 4, 5, 6, 8, 10]})
    if name == "Decision Tree":     return gs(DecisionTreeRegressor(random_state=SEED),
                                              {"max_depth": [2, 3, 4, 6, 8, 12, None], "min_samples_leaf": [1, 2, 4]})
    if name == "XGBoost":           return gs(XGBRegressor(n_estimators=400, random_state=SEED, n_jobs=1, verbosity=0,
                                                           tree_method="hist", device=XGB_DEVICE),
                                              {"max_depth": [2, 3, 6], "learning_rate": [0.03, 0.1]})
    if name == "LightGBM":          return gs(LGBMRegressor(n_estimators=400, random_state=SEED, n_jobs=1, verbose=-1, min_child_samples=3),
                                              {"num_leaves": [4, 8, 16], "learning_rate": [0.03, 0.1]})
    if name == "CatBoost":          return gs(CatBoostRegressor(iterations=400, random_seed=SEED, verbose=0, thread_count=1,
                                                                allow_writing_files=False, task_type=CAT_TASK),
                                              {"depth": [3, 5], "learning_rate": [0.03, 0.1]})
    if name == "Gaussian Process":  return make_gp_pipeline(n_feat)
    if name == "ANN":               return TorchRegressor("ann", NN_EPOCHS, seed=SEED, device=TORCH_DEVICE)
    if name == "CNN":               return TorchRegressor("cnn", NN_EPOCHS, seed=SEED, device=TORCH_DEVICE)
    raise ValueError(name)

def fit_model(est, X, y, groups=None):
    if isinstance(est, GridSearchCV) and groups is not None:
        return est.fit(X, y, groups=groups)
    return est.fit(X, y)

def describe_hparams(name, est):
    if isinstance(est, GridSearchCV):
        fmt = lambda v: f"{v:.4g}" if isinstance(v, float) else str(v)
        return ", ".join(f"{k.split('__')[-1]}={fmt(v)}" for k, v in est.best_params_.items())
    if name == "Gaussian Process":
        return str(est[-1].kernel_)
    if isinstance(est, TorchRegressor):
        return SEARCH_SPACE[name].replace("fixed: ", "") + f"; {est.epochs} epochs; final train MSE (std. units) {est.final_loss_:.4f}"
    return "none"

def base_estimator(est):
    return est.best_estimator_ if isinstance(est, GridSearchCV) else est

print("Helpers ready.")

## 2. Google Drive mounting
Mounts Drive, then creates any missing folders of the project tree. Existing files are never deleted.
Outside Colab (for testing) the same tree is created in a local folder instead.

In [ ]:
with step("2. Google Drive mounting"):
    DRIVE_OK = False
    if IN_COLAB:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            BASE_ROOT = Path("/content/drive/MyDrive"); DRIVE_OK = True
        except Exception as e:
            BASE_ROOT = Path("/content")
            note(f"Google Drive could not be mounted ({e}). Outputs go to /content/{PROJECT_NAME} and are LOST when the runtime ends.")
    else:
        BASE_ROOT = Path(os.environ.get("KOH_OUTPUT_ROOT", Path.cwd() / "local_output"))
        note(f"Not running in Colab: outputs are written to {BASE_ROOT / PROJECT_NAME} instead of Google Drive.")

    BASE = BASE_ROOT / PROJECT_NAME
    FIG_FOLDERS = ["exploratory", "model_comparison", "prediction", "uncertainty", "explainability", "experimental_design", "publication"]
    DIRS = {"base": BASE, "data": BASE / "data", "models": BASE / "models", "results": BASE / "results", "figures": BASE / "figures"}
    FIG_DIRS = {f: BASE / "figures" / f for f in FIG_FOLDERS}
    created = []
    for d in list(DIRS.values()) + list(FIG_DIRS.values()):
        if not d.exists():
            d.mkdir(parents=True, exist_ok=True); created.append(str(d.relative_to(BASE_ROOT)))
    print("Project folder :", BASE, "(Google Drive)" if DRIVE_OK else "")
    print("Created        :", ", ".join(created) if created else "nothing - all folders already existed")

## 3. Dataset loading
The lab files are read from `MyDrive/KoH_Surface_Area_Research/data/`. On the first run they are copied there from the
GitHub repository (cloned automatically). **An existing file in Drive is never overwritten**, so the Drive copy is your
master copy: add new experiments to it and re-run the notebook.

In [ ]:
def load_raw(path):
    """Parse the whitespace-separated lab file (Yield present only on some rows). Header / blank lines are skipped."""
    rows = []
    with open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            p = line.split()
            if not p or not re.fullmatch(r"\d+(\.\d+)?", p[0]):
                continue
            v = [float(x) for x in p][:5]
            rows.append(v + [np.nan] * (5 - len(v)))
    return pd.DataFrame(rows, columns=["IR", "Act_Temp", "Act_Time", "Sur_Area", "Yield"])

def locate_repo():
    for c in (Path.cwd(), Path.cwd().parent, Path("/content") / REPO_NAME):
        if (c / "data" / DATA_FILE_RAW).exists():
            return c
    target = (Path("/content") if IN_COLAB else Path.cwd()) / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(target)], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr); return None
    return target

with step("3. Dataset loading"):
    REPO_DIR = locate_repo()
    print("Repository     :", REPO_DIR if REPO_DIR else "NOT available (offline?)")
    for fn in (DATA_FILE_RAW, DATA_FILE_CSV):
        dst = DIRS["data"] / fn
        src = REPO_DIR / "data" / fn if REPO_DIR else None
        if dst.exists():
            same = src is not None and src.exists() and file_hash(src) == file_hash(dst)
            print(f"{fn:24s}: using existing Drive copy" + ("" if same else "  (differs from the repository copy - the Drive copy is used)"))
        elif src is not None and src.exists():
            shutil.copy2(src, dst); print(f"{fn:24s}: copied from the repository to Drive")
        else:
            print(f"{fn:24s}: NOT FOUND")
    RAW_PATH = DIRS["data"] / DATA_FILE_RAW
    if not RAW_PATH.exists():
        raise FileNotFoundError(f"Upload {DATA_FILE_RAW} to {DIRS['data']} and run the notebook again.")
    df_raw = load_raw(RAW_PATH)
    DATA_HASH = file_hash(RAW_PATH)
    CSV_PATH = DIRS["data"] / DATA_FILE_CSV
    df_csv = pd.read_csv(CSV_PATH) if CSV_PATH.exists() else None
    SAVED_RESULTS = REPO_DIR / "results" if REPO_DIR and (REPO_DIR / "results").exists() else None
    print(f"\nLoaded {len(df_raw)} rows from {RAW_PATH.name} (sha256 {DATA_HASH})")
    print("Saved repository results:", SAVED_RESULTS if SAVED_RESULTS else "not available")
    display(df_raw.head(8))

## 4. Dataset validation and preprocessing
Automatic checks (missing values, ranges, duplicates, full-factorial completeness, raw-file vs CSV agreement, yield
coverage). Then the modelling table for the chosen `TARGET` is built - **no values are imputed**: rows without a measured
target are excluded.

In [ ]:
with step("4. Dataset validation and preprocessing"):
    checks = []
    def chk(name, ok, detail, level="FAIL"):
        checks.append(dict(check=name, result="PASS" if ok else level, detail=detail))
    chk("Required columns", list(df_raw.columns) == ["IR", "Act_Temp", "Act_Time", "Sur_Area", "Yield"], ", ".join(df_raw.columns))
    chk("Surface area present on every row", df_raw.Sur_Area.notna().all(), f"{df_raw.Sur_Area.isna().sum()} missing")
    chk("Surface area > 0", (df_raw.Sur_Area.dropna() > 0).all(), f"min = {df_raw.Sur_Area.min():g}")
    for col, lo, hi in (("IR", 0, 6), ("Act_Temp", 800, 1200), ("Act_Time", 0, 90)):
        chk(f"{col} within [{lo}, {hi}]", df_raw[col].between(lo, hi).all(), f"observed {df_raw[col].min():g}-{df_raw[col].max():g}", "WARN")
    dup = df_raw.duplicated(["IR", "Act_Temp", "Act_Time"], keep=False)
    chk("No duplicated conditions", not dup.any(), f"{dup.sum()} rows share a condition (possible replicates)", "WARN")
    levels = {c: sorted(df_raw[c].unique()) for c in ["IR", "Act_Temp", "Act_Time"]}
    n_grid = np.prod([len(v) for v in levels.values()])
    chk("Full-factorial grid complete", len(df_raw.drop_duplicates(["IR", "Act_Temp", "Act_Time"])) == n_grid,
        f"{len(df_raw)} rows vs {n_grid} = " + " × ".join(str(len(v)) for v in levels.values()), "WARN")
    if df_csv is not None:
        same = df_csv.shape == df_raw.shape and np.allclose(df_csv.fillna(-1).values, df_raw.fillna(-1).values)
        chk("Raw file and cleaned CSV agree", same, f"CSV {df_csv.shape}, raw {df_raw.shape}", "WARN")
    yc = df_raw.groupby("IR").Yield.count()
    chk("Yield coverage", True, "measured yields per IR: " + ", ".join(f"IR{int(k)}={v}" for k, v in yc.items()), "WARN")
    tie = df_raw[df_raw.Sur_Area.duplicated(keep=False)].sort_values("Sur_Area")
    chk("Identical surface-area values at different conditions", tie.empty,
        f"{tie.Sur_Area.nunique()} values repeat - verify against lab notebooks", "WARN")
    CHECKS = pd.DataFrame(checks); display(CHECKS); save_table(CHECKS, "data_validation_checks", index=False)
    if not tie.empty:
        print("Repeated surface-area values (informational):"); display(tie)

    # ---- modelling table for the chosen target ----
    if TARGET == "Sur_Area":
        FEATURES = ["IR", "Act_Temp", "Act_Time"]
        data = df_raw.dropna(subset=["Sur_Area"]).copy()
        TARGET_DESC = "Surface area, all impregnation ratios"
    elif TARGET == "Yield":
        FEATURES = ["Act_Temp", "Act_Time"]
        data = df_raw[(df_raw.IR == 4) & df_raw.Yield.notna()].copy()
        TARGET_DESC = "Yield at IR = 4 (only IR with a full yield grid)"
        note("Yield models use IR = 4 only (35 points). IR 1, 2, 3, 5, 6 have a single yield point each and IR 0 none, "
             "so yield modelling for other IRs is UNAVAILABLE.")
    else:
        raise ValueError("TARGET must be 'Sur_Area' or 'Yield'")
    data["IR"] = data["IR"].astype(int)
    data = data.reset_index().rename(columns={"index": "row_id"})
    UNIT = SA_UNIT if TARGET == "Sur_Area" else YIELD_UNIT
    IR_LEVELS = sorted(data.IR.unique())
    print(f"\nTARGET = {TARGET} ({TARGET_DESC}); features = {FEATURES}; n = {len(data)} measured samples")
    if TARGET == "Sur_Area":
        note(f"Surface-area unit '{SA_UNIT}' is an ASSUMPTION (BET); the lab file does not state it. Confirm before publishing.")

## 5. Exploratory data analysis
*Paper analogue: Fig. 6 (scatter matrix).* Plus the measured grid as annotated heat maps and raw response curves.

In [ ]:
with step("5. Exploratory data analysis"):
    summ = data[FEATURES + [TARGET]].describe().T; display(summ); save_table(summ, "descriptive_statistics")
    if "IR" in FEATURES:
        display(data.groupby("IR")[TARGET].agg(["count", "min", "max", "mean"]).round(1))

    # Fig: scatter matrix (paper Fig. 6)
    cols = FEATURES + [TARGET]
    g = sns.pairplot(data[cols].rename(columns={c: short(c) for c in cols}), diag_kind="kde", height=1.9,
                     plot_kws=dict(s=14, alpha=0.7, edgecolor="none", color="#2b6cb0"), diag_kws=dict(fill=True, color="#2b6cb0"))
    g.figure.suptitle(f"Scatter matrix of measured variables (n = {len(data)}; {TARGET_DESC})", y=1.02)
    save_fig(g.figure, "F01_scatter_matrix", "exploratory", "Scatter matrix", "Pairwise scatter plots with KDE diagonals of the inputs and target.",
             "Fig. 6", len(data), "descriptive")

    # Fig: measured grid as annotated heat maps
    panels = IR_LEVELS if "IR" in FEATURES else [4]
    ncol = 4 if len(panels) > 1 else 1
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.1 * nrow), squeeze=False, constrained_layout=True)
    vmin, vmax = data[TARGET].min(), data[TARGET].max()
    for ax, ir in zip(axs.ravel(), panels):
        d = data[data.IR == ir]
        piv = d.pivot_table(index="Act_Time", columns="Act_Temp", values=TARGET)
        sns.heatmap(piv, ax=ax, annot=True, fmt=".0f" if TARGET == "Sur_Area" else ".1f", cmap="viridis", vmin=vmin, vmax=vmax,
                    cbar=False, annot_kws={"size": 6}, linewidths=0.3, linecolor="white")
        ax.invert_yaxis(); ax.set_title(f"IR = {ir} (n = {len(d)})"); ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)"); ax.grid(False)
    for ax in axs.ravel()[len(panels):]:
        ax.axis("off")
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap="viridis"), ax=axs, shrink=0.7, label=label(TARGET))
    fig.suptitle(f"Measured {short(TARGET).lower()} on the experimental grid")
    save_fig(fig, "F02_measured_grid", "exploratory", "Measured grid", "Annotated heat maps of every measured value (time × temperature) per IR.",
             "— (project)", len(data), "descriptive")

    # Fig: raw response curves
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 2.9 * nrow), squeeze=False, sharey=True, constrained_layout=True)
    cmapT = plt.get_cmap("plasma", len(levels["Act_Temp"]) + 1)
    for ax, ir in zip(axs.ravel(), panels):
        d = data[data.IR == ir]
        for j, T in enumerate(sorted(d.Act_Temp.unique())):
            dd = d[d.Act_Temp == T].sort_values("Act_Time")
            ax.plot(dd.Act_Time, dd[TARGET], "-o", ms=3, color=cmapT(j), label=f"{T:g} °C")
        ax.set_title(f"IR = {ir}"); ax.set_xlabel(label("Act_Time")); ax.set_ylabel(label(TARGET))
    for ax in axs.ravel()[len(panels):]:
        ax.axis("off")
    axs.ravel()[0].legend(title="Temperature", fontsize=6.5)
    fig.suptitle(f"Measured {short(TARGET).lower()} versus activation time (lines join measured points only)")
    save_fig(fig, "F03_measured_curves", "exploratory", "Measured response curves", "Measured target vs time for each temperature and IR.",
             "— (project)", len(data), "descriptive")

## 6. Outlier analysis
*Paper analogue: Fig. 8 (Monte Carlo outlier detection + box plot).*

**Method (Monte Carlo cross-prediction).** A Gaussian process with hyper-parameters fixed at their full-data values is
refitted `MCOD_ITERATIONS` times on random 75 % subsets; every sample collects its out-of-subset prediction errors.
Samples whose standardised mean error lies beyond ±2 SD (the paper's bounds) are **flagged for checking against the lab
notebook - they are not deleted** unless `REMOVE_OUTLIERS = True`. With no replicate measurements, a flag can mean either a
measurement problem or genuine sharp behaviour (e.g. pore collapse at long times).

In [ ]:
with step("6. Outlier analysis"):
    Xd, yd = data[FEATURES].to_numpy(float), data[TARGET].to_numpy(float)
    REF_GP = make_gp_pipeline(len(FEATURES)).fit(Xd, yd)          # diagnostic GP on all data
    REF_KERNEL = REF_GP[-1].kernel_
    Zd = REF_GP[0].transform(Xd)
    rng = np.random.default_rng(SEED); n = len(yd); m_sub = int(0.75 * n)
    s1, s2, cnt = np.zeros(n), np.zeros(n), np.zeros(n)
    for _ in tqdm(range(MCOD_ITERATIONS), desc="Monte Carlo sub-models"):
        tr = rng.choice(n, m_sub, replace=False); te = np.setdiff1d(np.arange(n), tr)
        gpm = GaussianProcessRegressor(REF_KERNEL, optimizer=None, normalize_y=True).fit(Zd[tr], yd[tr])
        e = yd[te] - gpm.predict(Zd[te]); s1[te] += e; s2[te] += e ** 2; cnt[te] += 1
    data["mc_mean_err"] = s1 / cnt
    data["mc_sd_err"] = np.sqrt(np.maximum(s2 / cnt - (s1 / cnt) ** 2, 0))
    data["mc_z"] = (data.mc_mean_err - data.mc_mean_err.mean()) / data.mc_mean_err.std()
    data["mc_flag"] = data.mc_z.abs() > 2
    FLAGS = data[data.mc_flag].sort_values("mc_z", key=np.abs, ascending=False)
    print(f"{len(FLAGS)} of {n} samples flagged (|z| > 2). Each sample was out-of-subset on average {cnt.mean():.0f} times.")
    if REMOVE_OUTLIERS and len(FLAGS):
        data_model = data[~data.mc_flag].reset_index(drop=True)
        note(f"REMOVE_OUTLIERS=True: {len(FLAGS)} flagged rows were excluded from modelling.")
    else:
        data_model = data.reset_index(drop=True)
    print(f"Rows used for modelling: {len(data_model)}" + ("" if REMOVE_OUTLIERS else " (flagged rows kept - REMOVE_OUTLIERS = False)"))
    display(FLAGS[["row_id"] + FEATURES + [TARGET, "mc_mean_err", "mc_sd_err", "mc_z"]].round(2))
    save_table(data[["row_id"] + FEATURES + [TARGET, "mc_mean_err", "mc_sd_err", "mc_z", "mc_flag"]], "monte_carlo_outliers", index=False)
    if SAVED_RESULTS is not None and (SAVED_RESULTS / "data_quality_flags.csv").exists() and TARGET == "Sur_Area":
        prev = pd.read_csv(SAVED_RESULTS / "data_quality_flags.csv")
        both = prev.merge(FLAGS[FEATURES], on=FEATURES, how="left", indicator=True)
        print(f"Repository flags (closed-form LOO, results/data_quality_flags.csv): {len(prev)} rows; "
              f"{(both._merge == 'both').sum()} of them are also flagged here.")

    # Fig: histogram with ±2 SD bounds (paper Fig. 8A) + mean vs SD of errors
    fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.9))
    axs[0].hist(data.mc_z, bins=30, color="#9ecae1", edgecolor="#3182bd")
    for v, c, lab in ((-2, "#c53030", "Lower bound (−2 SD)"), (2, "#2f855a", "Upper bound (+2 SD)")):
        axs[0].axvline(v, color=c, ls="--", lw=1.2, label=lab)
    axs[0].set_xlabel("Standardised mean Monte Carlo prediction error (z)"); axs[0].set_ylabel("Frequency (samples)")
    axs[0].set_title(f"A  Monte Carlo outlier detection\n({MCOD_ITERATIONS} sub-models, n = {n})"); axs[0].legend()
    axs[1].scatter(data.mc_mean_err, data.mc_sd_err, s=14, c=np.where(data.mc_flag, "#c53030", "#3182bd"))
    for i, (_, r) in enumerate(FLAGS.sort_values("mc_mean_err").iterrows()):
        txt = ", ".join(f"{r[f]:g}" for f in FEATURES)
        axs[1].annotate(f"({txt})", (r.mc_mean_err, r.mc_sd_err), fontsize=5.5, xytext=(4, 5 if i % 2 == 0 else -9),
                        textcoords="offset points")
    axs[1].set_xlabel(f"Mean prediction error ({UNIT})"); axs[1].set_ylabel(f"SD of prediction error ({UNIT})")
    axs[1].set_title("B  Mean vs spread of Monte Carlo errors\n(red = flagged; labels = " + ", ".join(short(f) for f in FEATURES) + ")")
    fig.tight_layout()
    save_fig(fig, "F04_monte_carlo_outliers", "exploratory", "Monte Carlo outlier detection",
             "Histogram of standardised Monte Carlo cross-prediction errors with ±2 SD bounds; mean vs SD of errors.",
             "Fig. 8A", n, f"Monte Carlo cross-prediction, {MCOD_ITERATIONS} random 75 % subsets")

    # Fig: box plots of standardised variables (paper Fig. 8B)
    zdf = (data[FEATURES + [TARGET]] - data[FEATURES + [TARGET]].mean()) / data[FEATURES + [TARGET]].std()
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.boxplot([zdf[c] for c in zdf.columns], patch_artist=True,
               boxprops=dict(facecolor="#bee3f8", edgecolor="#2c5282"), medianprops=dict(color="#c53030"), flierprops=dict(marker="o", ms=3))
    ax.set_xticks(range(1, len(zdf.columns) + 1)); ax.set_xticklabels([short(c) for c in zdf.columns])
    ax.set_ylabel("Standardised value (z-score)"); ax.set_title(f"Data dispersion (n = {n})")
    save_fig(fig, "F05_boxplots", "exploratory", "Box plots of standardised variables", "Box plots of z-scored inputs and target.",
             "Fig. 8B", n, "descriptive")

## 7. Correlation analysis
*Paper analogue: Fig. 7.* Pearson (linear) and Spearman (monotonic) coefficients with two-sided p-values.
In a full-factorial design the inputs are uncorrelated by construction, so the informative entries are input ↔ target.

In [ ]:
def corr_with_p(df, method):
    cols = df.columns; r = pd.DataFrame(index=cols, columns=cols, dtype=float); p = r.copy()
    f = stats.pearsonr if method == "pearson" else stats.spearmanr
    for a in cols:
        for b in cols:
            res = f(df[a], df[b]); r.loc[a, b] = res[0]; p.loc[a, b] = res[1]
    return r, p

def corr_figure(df, title, key, n_note):
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
    out = {}
    for ax, method in zip(axs, ("pearson", "spearman")):
        r, p = corr_with_p(df, method); out[method] = (r, p)
        ann = r.round(2).astype(str) + "\n" + p.map(lambda v: "p<0.001" if v < 1e-3 else f"p={v:.3f}")
        sns.heatmap(r, annot=ann, fmt="", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax, square=True, annot_kws={"size": 7},
                    xticklabels=[short(c) for c in df.columns], yticklabels=[short(c) for c in df.columns],
                    cbar_kws={"label": f"{method.title()} coefficient"})
        ax.set_title(f"{method.title()} correlation"); ax.grid(False)
    fig.suptitle(title); fig.tight_layout()
    save_fig(fig, key, "exploratory", title, "Pearson and Spearman correlation heat maps with p-values.", "Fig. 7", n_note, "descriptive")
    return out

with step("7. Correlation analysis"):
    cols = FEATURES + [TARGET]
    CORR = corr_figure(data_model[cols], f"Correlation analysis (n = {len(data_model)}; {TARGET_DESC})", "F06_correlation_heatmap", len(data_model))
    save_table(CORR["pearson"][0], "pearson_r"); save_table(CORR["pearson"][1], "pearson_p")
    save_table(CORR["spearman"][0], "spearman_r"); save_table(CORR["spearman"][1], "spearman_p")
    sub = df_raw[(df_raw.IR == 4) & df_raw.Yield.notna()][["Act_Temp", "Act_Time", "Sur_Area", "Yield"]]
    corr_figure(sub, f"Surface area and yield at IR = 4 (n = {len(sub)})", "F07_correlation_IR4_SA_yield", len(sub))

## 8. Machine learning model loading / training
**Split (Part A).** Random 70 / 15 / 15 train / validation / test split of the measured samples (stratified by IR for
surface area), fixed by `SEED`. The validation set is used only to *select* the best model; the test set is reported
once and never used for any decision.

**Tuning without leakage.** Hyper-parameters are chosen by `INNER_CV_FOLDS`-fold cross-validation **inside the training
set only**; scaling is fitted inside each pipeline on training folds only. The ANN and CNN use the fixed architectures
of the reference paper (Table 1) and are trained on the GPU when one is available.

**Caching.** Fitted models and their predictions are saved to `models/` and `results/` in Drive. On a re-run they are
loaded instead of retrained if the data file, split and configuration are unchanged (`FORCE_RETRAIN = True` overrides).

Note: the experimental grid is regular, so a random split asks the models to *interpolate* between neighbouring grid
points. This is the reference paper's protocol; the harder *extrapolation* tests (unseen temperature / time) follow in
Section 9b.

In [ ]:
with step("8. Machine learning model loading / training"):
    X_all = data_model[FEATURES].to_numpy(float); y_all = data_model[TARGET].to_numpy(float)
    idx = np.arange(len(y_all))
    strat = data_model.IR.to_numpy() if "IR" in FEATURES else None
    f_tr, f_va, f_te = SPLIT_FRACTIONS
    idx_tr, idx_tmp = train_test_split(idx, train_size=f_tr, random_state=SEED, stratify=strat)
    idx_va, idx_te = train_test_split(idx_tmp, test_size=f_te / (f_va + f_te), random_state=SEED,
                                      stratify=strat[idx_tmp] if strat is not None else None)
    SETS = {"Train": np.sort(idx_tr), "Validation": np.sort(idx_va), "Test": np.sort(idx_te)}
    assert not (set(idx_tr) & set(idx_va) or set(idx_tr) & set(idx_te) or set(idx_va) & set(idx_te)), "split overlap!"
    SPLIT_DESC = (f"random split, seed {SEED}: train n = {len(idx_tr)}, validation n = {len(idx_va)}, test n = {len(idx_te)}"
                  + (" (stratified by IR)" if strat is not None else ""))
    print(SPLIT_DESC)
    split_df = data_model[["row_id"] + FEATURES + [TARGET]].copy()
    split_df["set"] = ""
    for s, ix in SETS.items():
        split_df.loc[ix, "set"] = s
    save_table(split_df, "partA_split", index=False)

    cache_key = hashlib.sha256(json.dumps(dict(t=TARGET, h=DATA_HASH, seed=SEED, split=SPLIT_FRACTIONS, rm=REMOVE_OUTLIERS,
                                               v=CONFIG_VERSION, ep=NN_EPOCHS, cv=INNER_CV_FOLDS, models=MODEL_NAMES,
                                               idx=[int(i) for i in idx_tr])).encode()).hexdigest()[:16]
    bundle_path = DIRS["models"] / f"{TARGET}_partA_models.joblib"
    FITTED, PREDS, FIT_TIME, FAILED = {}, {}, {}, {}
    loaded = False
    if bundle_path.exists() and not FORCE_RETRAIN:
        try:
            # joblib uses pickle: only load this cache because this notebook wrote it to YOUR Drive. Never load .joblib
            # files from someone else; delete the models/ folder (or set FORCE_RETRAIN=True) if in doubt.
            B = joblib.load(bundle_path)
            if B.get("cache_key") == cache_key:
                FITTED, PREDS, FIT_TIME = B["models"], B["preds"], B["fit_time"]; loaded = True
                print(f"Loaded {len(FITTED)} fitted models and their saved predictions from {bundle_path.name} "
                      f"(trained {B['created']}).")
            else:
                print("Cached models exist but data/configuration changed -> retraining.")
        except Exception as e:
            print(f"Could not load cached models ({e}) -> retraining.")
    if not loaded:
        inner = KFold(INNER_CV_FOLDS, shuffle=True, random_state=SEED)
        for name in tqdm(MODEL_NAMES, desc="Training 15 models"):
            set_seeds()
            est = build_model(name, len(FEATURES), inner)
            t0 = time.time()
            try:
                fit_model(est, X_all[SETS["Train"]], y_all[SETS["Train"]])
            except Exception as e:
                FAILED[name] = f"{type(e).__name__}: {e}"; print(f"  {name} FAILED: {e}"); continue
            FIT_TIME[name] = time.time() - t0; FITTED[name] = est
            PREDS[name] = {s: est.predict(X_all[ix]) for s, ix in SETS.items()}
        joblib.dump(dict(cache_key=cache_key, models=FITTED, preds=PREDS, fit_time=FIT_TIME, features=FEATURES, target=TARGET,
                         data_hash=DATA_HASH, created=f"{datetime.datetime.now():%Y-%m-%d %H:%M}", versions=VERSIONS), bundle_path)
        print(f"Saved models + predictions to {bundle_path}")
    MODELS_OK = [m for m in MODEL_NAMES if m in PREDS]
    if FAILED:
        note("Models that failed to train and are excluded: " + "; ".join(f"{k} ({v})" for k, v in FAILED.items()))
    pred_rows = []
    for name in MODELS_OK:
        for s, ix in SETS.items():
            pred_rows.append(pd.DataFrame({"model": name, "set": s, "row_id": data_model.row_id.values[ix],
                                           **{f: X_all[ix, j] for j, f in enumerate(FEATURES)},
                                           "actual": y_all[ix], "predicted": PREDS[name][s]}))
    PRED_LONG = pd.concat(pred_rows, ignore_index=True); save_table(PRED_LONG, "partA_predictions", index=False)
    print(f"{len(MODELS_OK)} models available: {', '.join(MODELS_OK)}")

## 9. Model benchmarking
**9a - Table 1 (tuned structure) and Table 2 (metrics)** in the paper's format: R², MSE, RMSE, residual SD (σ), MAE and
MRD % (mean *absolute* relative deviation, 100·mean|y−ŷ|/y) for the train, validation and test sets.
The best model is the one with the highest **validation** R².

In [ ]:
with step("9a. Benchmark tables"):
    TABLE1 = pd.DataFrame([dict(model=m, search_space=SEARCH_SPACE[m], selected=describe_hparams(m, FITTED[m]),
                                fit_time_s=round(FIT_TIME.get(m, np.nan), 2)) for m in MODELS_OK]).set_index("model")
    display(Markdown("**Table 1 - tuned structure of each model** (selected by inner CV on the training set)")); display(TABLE1)
    save_table(TABLE1, "table1_hyperparameters")
    rows = [dict(model=m, set=s, **metrics(y_all[ix], PREDS[m][s])) for m in MODELS_OK for s, ix in SETS.items()]
    METRICS_LONG = pd.DataFrame(rows); save_table(METRICS_LONG, "table2_metrics_long", index=False)
    TABLE2 = METRICS_LONG.pivot(index="model", columns="set", values=["R2", "MSE", "RMSE", "MRD"])
    TABLE2 = TABLE2.reindex(columns=["Train", "Validation", "Test"], level=1).loc[MODELS_OK]
    BEST = METRICS_LONG[METRICS_LONG.set == "Validation"].sort_values("R2", ascending=False).model.iloc[0]
    display(Markdown(f"**Table 2 - metrics ({SPLIT_DESC}; MSE in ({UNIT})²; RMSE in {UNIT})**  -  best by validation R²: **{BEST}**"))
    display(TABLE2.style.format("{:.3f}", subset=["R2"]).format("{:.1f}", subset=["MSE", "RMSE", "MRD"])
            .highlight_max(subset=[("R2", "Validation")], color="#c6f6d5"))
    save_table(TABLE2.round(4), "table2_metrics")
    _flat = TABLE2.round(3).copy(); _flat.columns = [f"{a} {b}" for a, b in _flat.columns]
    try:
        _md = _flat.to_markdown()
    except ImportError:            # 'tabulate' missing -> plain text
        _md = _flat.to_string()
    (DIRS["results"] / f"{TARGET}_table2_metrics.md").write_text(_md, encoding="utf-8")

    order = METRICS_LONG[METRICS_LONG.set == "Validation"].sort_values("R2").model.tolist()
    fig, axs = plt.subplots(1, 2, figsize=(11, 5))
    h = 0.27
    for i, s in enumerate(["Train", "Validation", "Test"]):
        v = METRICS_LONG[METRICS_LONG.set == s].set_index("model").loc[order]
        axs[0].barh(np.arange(len(order)) + (i - 1) * h, v.R2, h, color=SET_COLORS[s], label=f"{s} (n = {len(SETS[s])})")
        axs[1].barh(np.arange(len(order)) + (i - 1) * h, v.RMSE, h, color=SET_COLORS[s])
    for ax in axs:
        ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
    axs[0].set_xlabel("R²"); axs[0].set_xlim(max(0, METRICS_LONG.R2.min() - 0.05), 1.0); axs[1].set_xlabel(f"RMSE ({UNIT})")
    axs[0].legend(loc="lower right", fontsize=7); axs[1].set_yticklabels([])
    fig.suptitle(f"Model benchmark - {TARGET_DESC} ({SPLIT_DESC})", fontsize=9); fig.tight_layout()
    save_fig(fig, "F08_benchmark_random_split", "model_comparison", "Benchmark (random split)",
             "R² and RMSE of the 15 models on train / validation / test.", "Table 2", len(y_all), SPLIT_DESC)

**9b - Grouped (extrapolation) benchmark - project-specific.** Each interior temperature (`GROUP_TEMPS`) or interior time
(`GROUP_TIMES`) is held out **completely**, all 15 models are re-tuned on the remaining data (inner cross-validation grouped
by the same variable, so tuning also never sees the held-out level) and predict the unseen level. This mimics the real
question "what happens at a temperature/time we did not run?". Results are compared with the saved repository benchmark
(`results/benchmark_SA_long.csv`) where available.

In [ ]:
def grouped_splits(df):
    if "Act_Temp" in FEATURES:
        for T in GROUP_TEMPS:
            if (df.Act_Temp == T).any():
                yield "LOTemp", f"{T:g} °C", np.where(df.Act_Temp != T)[0], np.where(df.Act_Temp == T)[0], df.Act_Temp.to_numpy()
    for t in GROUP_TIMES:
        if (df.Act_Time == t).any():
            yield "LOTime", f"{t:g} min", np.where(df.Act_Time != t)[0], np.where(df.Act_Time == t)[0], df.Act_Time.to_numpy()

SCHEME_NAME = {"LOTemp": "Leave-one-temperature-out", "LOTime": "Leave-one-time-out"}

with step("9b. Grouped benchmark (leave-one-temperature-out / leave-one-time-out)"):
    if not RUN_GROUPED_BENCHMARK:
        note("Grouped benchmark skipped (RUN_GROUPED_BENCHMARK = False).")
    else:
        gkey = hashlib.sha256(f"{cache_key}{GROUP_TEMPS}{GROUP_TIMES}".encode()).hexdigest()[:16]
        gpath = DIRS["results"] / f"{TARGET}_grouped_oof_predictions.csv"
        GOOF = None
        if gpath.exists() and not FORCE_RETRAIN:
            tmp = pd.read_csv(gpath)
            if "cache_key" in tmp and (tmp.cache_key == gkey).all():
                GOOF = tmp; print(f"Loaded saved grouped out-of-fold predictions ({len(GOOF)} rows) from {gpath.name}")
        if GOOF is None:
            folds = list(grouped_splits(data_model)); recs = []
            bar = tqdm(total=len(folds) * len(MODELS_OK), desc="Grouped benchmark")
            for scheme, fold, tr, te, grp in folds:
                ng = len(np.unique(grp[tr]))
                inner = GroupKFold(n_splits=min(INNER_CV_FOLDS, ng))
                for name in MODELS_OK:
                    set_seeds(); est = build_model(name, len(FEATURES), inner)
                    try:
                        fit_model(est, X_all[tr], y_all[tr], groups=grp[tr]); p = est.predict(X_all[te])
                        recs.append(pd.DataFrame(dict(scheme=scheme, fold=fold, model=name, row_id=data_model.row_id.values[te],
                                                      **{f: X_all[te, j] for j, f in enumerate(FEATURES)}, actual=y_all[te], predicted=p)))
                    except Exception as e:
                        print(f"  {name} / {fold} failed: {e}")
                    bar.update(1)
            bar.close()
            GOOF = pd.concat(recs, ignore_index=True); GOOF["cache_key"] = gkey; GOOF.to_csv(gpath, index=False)
        GMET = pd.DataFrame([dict(scheme=s, model=m, **metrics(g.actual, g.predicted)) for (s, m), g in GOOF.groupby(["scheme", "model"])])
        save_table(GMET, "grouped_benchmark_metrics", index=False)
        GTAB = GMET.pivot(index="model", columns="scheme", values=["R2", "RMSE", "MRD"])
        sort_col = ("R2", "LOTemp") if ("R2", "LOTemp") in GTAB.columns else GTAB.columns[0]
        GTAB = GTAB.sort_values(sort_col, ascending=False)
        display(Markdown(f"**Grouped benchmark** (out-of-fold, pooled over held-out levels; n per scheme: "
                         + ", ".join(f"{s} = {g.row_id.nunique()}" for s, g in GOOF.groupby('scheme')) + ")"))
        display(GTAB.round(3))

        # comparison with saved repository benchmark
        if SAVED_RESULTS is not None and (SAVED_RESULTS / "benchmark_SA_long.csv").exists() and TARGET == "Sur_Area":
            prev = pd.read_csv(SAVED_RESULTS / "benchmark_SA_long.csv")
            mp = {"Linear Regression": "Linear", "SVR": "SVR", "Random Forest": "RandomForest", "Gradient Boosting": "GradBoost",
                  "KNN": "KNN", "XGBoost": "XGBoost", "LightGBM": "LightGBM", "CatBoost": "CatBoost",
                  "Gaussian Process": "GPR-Matern52-ARD", "ANN": "MLP"}
            comp = []
            for m, pm in mp.items():
                for s in ("LOTemp", "LOTime"):
                    a = GMET[(GMET.model == m) & (GMET.scheme == s)].R2; b = prev[(prev.model == pm) & (prev.scheme == s)].R2
                    if len(a) and len(b):
                        comp.append(dict(model=m, scheme=s, R2_this_notebook=float(a.iloc[0]), R2_saved_repository=float(b.iloc[0]),
                                         repository_model=pm))
            if comp:
                COMP = pd.DataFrame(comp); COMP["difference"] = COMP.R2_this_notebook - COMP.R2_saved_repository
                display(Markdown("**Comparison with the saved repository benchmark** (differences come from different tuning grids, "
                                 "grouped inner CV here vs random inner CV there, and library versions; ANN ≠ repository MLP architecture)"))
                display(COMP.round(3)); save_table(COMP, "grouped_vs_repository", index=False)

        fig, axs = plt.subplots(1, 2, figsize=(11, 5.2))
        order = GTAB.index[::-1].tolist(); h = 0.38
        for i, (s, c) in enumerate((("LOTemp", "#c53030"), ("LOTime", "#2b6cb0"))):
            if ("R2", s) not in GTAB.columns:
                continue
            nn_ = GOOF[GOOF.scheme == s].row_id.nunique()
            axs[0].barh(np.arange(len(order)) + (i - 0.5) * h, GTAB.loc[order, ("R2", s)], h, color=c, label=f"{SCHEME_NAME[s]} (n = {nn_})")
            axs[1].barh(np.arange(len(order)) + (i - 0.5) * h, GTAB.loc[order, ("RMSE", s)], h, color=c)
        axs[0].set_yticks(range(len(order))); axs[0].set_yticklabels(order); axs[1].set_yticks(range(len(order))); axs[1].set_yticklabels([])
        axs[0].set_xlabel("R² (out-of-fold)"); axs[0].set_xlim(max(-0.2, np.nanmin(GTAB["R2"].values) - 0.05), 1)
        axs[1].set_xlabel(f"RMSE ({UNIT})"); axs[0].legend(loc="lower right", fontsize=7)
        fig.suptitle(f"Prediction at unseen conditions - held-out temperatures {GROUP_TEMPS} °C / times {GROUP_TIMES} min "
                     "(nested, grouped tuning)", fontsize=9)
        fig.tight_layout()
        save_fig(fig, "F09_grouped_benchmark", "model_comparison", "Leave-one-temperature/time-out benchmark",
                 "R² and RMSE of all models when a whole temperature or time level is unseen.", "— (project)",
                 len(data_model), "leave-one-temperature-out and leave-one-time-out, nested grouped tuning")

## 10. Training, validation and testing evaluation
Generalisation gap (train vs validation vs test R²) for every model and residual diagnostics of the selected model.

In [ ]:
with step("10. Training, validation and testing evaluation"):
    r2 = METRICS_LONG.pivot(index="model", columns="set", values="R2").loc[MODELS_OK, ["Train", "Validation", "Test"]]
    r2["gap (train-test)"] = r2.Train - r2.Test
    fig, ax = plt.subplots(figsize=(6.5, 6))
    sns.heatmap(r2, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax, annot_kws={"size": 7}, cbar_kws={"label": "R² (gap column: difference)"})
    ax.set_title(f"R² by data set ({SPLIT_DESC})", fontsize=8); ax.grid(False); ax.set_ylabel("")
    save_fig(fig, "F10_r2_train_val_test", "model_comparison", "R² heat map train/validation/test",
             "R² of each model on each set and the train-test gap (over-fitting indicator).", "Table 2", len(y_all), SPLIT_DESC)

    fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.8))
    for s, ix in SETS.items():
        res = y_all[ix] - PREDS[BEST][s]
        axs[0].scatter(PREDS[BEST][s], res, s=14, alpha=0.7, color=SET_COLORS[s], label=f"{s} (n = {len(ix)})")
        axs[1].hist(res, bins=20, alpha=0.55, color=SET_COLORS[s], label=s)
    axs[0].axhline(0, color="k", lw=0.8); axs[0].set_xlabel(f"Predicted {short(TARGET).lower()} ({UNIT})"); axs[0].set_ylabel(f"Residual ({UNIT})")
    axs[1].set_xlabel(f"Residual ({UNIT})"); axs[1].set_ylabel("Count"); axs[0].legend(); axs[1].legend()
    fig.suptitle(f"Residual diagnostics of the selected model: {BEST}"); fig.tight_layout()
    save_fig(fig, "F11_residuals_best_model", "model_comparison", f"Residuals of {BEST}", "Residual vs predicted and residual histograms.",
             "—", len(y_all), SPLIT_DESC)

## 11. Taylor diagrams
*Paper analogue: Fig. 9.* Each model is placed by its correlation with the measurements (angle) and its standard deviation
normalised by that of the measurements (radius); grey dashed arcs are the centred RMS difference. The star is a perfect model.

In [ ]:
def taylor_panel(fig, pos, ref, preds, title):
    ax = fig.add_subplot(pos, projection="polar")
    sd_ref = np.std(ref); pts = []
    for name, p in preds.items():
        pts.append((name, np.std(p) / sd_ref, np.corrcoef(ref, p)[0, 1]))
    smax = max(1.5, 1.1 * max(s for _, s, _ in pts))
    ax.set_thetamin(0); ax.set_thetamax(90); ax.set_ylim(0, smax)
    ct = np.array([0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99])
    ax.set_thetagrids(np.degrees(np.arccos(ct)), labels=[f"{c:g}" for c in ct], fontsize=7)
    th, rr = np.meshgrid(np.linspace(0, np.pi / 2, 120), np.linspace(0, smax, 120))
    rms = np.sqrt(1 + rr ** 2 - 2 * rr * np.cos(th))
    cs = ax.contour(th, rr, rms, levels=np.arange(0.25, 2.01, 0.25), colors="0.55", linewidths=0.6, linestyles="--")
    ax.clabel(cs, fmt="%.2f", fontsize=6)
    ax.plot(np.linspace(0, np.pi / 2, 100), np.ones(100), "k:", lw=0.9)
    ax.plot(0, 1, "k*", ms=13, zorder=5)
    for name, s, r in pts:
        c, mk = MODEL_STYLE[name]
        ax.plot(np.arccos(np.clip(r, 0, 1)), s, marker=mk, color=c, ms=7, ls="", mec="k", mew=0.4, zorder=6)
    ax.set_title(title, pad=14, fontsize=9)
    ax.text(np.radians(48), smax * 1.13, "Correlation", fontsize=8, rotation=-42, ha="center")
    ax.text(0.5, -0.13, "Normalised standard deviation", transform=ax.transAxes, ha="center", va="top", fontsize=8)
    return ax

with step("11. Taylor diagrams"):
    fig = plt.figure(figsize=(15, 5.6))
    for i, (s, ix) in enumerate(SETS.items()):
        taylor_panel(fig, 131 + i, y_all[ix], {m: PREDS[m][s] for m in MODELS_OK}, f"{s} (n = {len(ix)})")
    handles = [Line2D([], [], marker="*", color="k", ls="", ms=11, label="Reference (measured)")] + \
              [Line2D([], [], marker=MODEL_STYLE[m][1], color=MODEL_STYLE[m][0], ls="", mec="k", mew=0.4, ms=7, label=m) for m in MODELS_OK]
    fig.legend(handles=handles, loc="upper center", ncol=8, frameon=False, bbox_to_anchor=(0.5, 0.12))
    fig.subplots_adjust(bottom=0.22, top=0.80, wspace=0.25)
    fig.suptitle(f"Taylor diagrams of regression models - {TARGET_DESC}\n{SPLIT_DESC}", fontsize=9)
    TAYLOR = pd.DataFrame([dict(model=m, set=s, corr=np.corrcoef(y_all[ix], PREDS[m][s])[0, 1],
                                norm_sd=np.std(PREDS[m][s]) / np.std(y_all[ix])) for m in MODELS_OK for s, ix in SETS.items()])
    save_table(TAYLOR, "taylor_statistics", index=False)
    save_fig(fig, "F12_taylor_diagrams", "model_comparison", "Taylor diagrams", "Correlation / normalised SD / centred RMS of each model per set.",
             "Fig. 9", len(y_all), SPLIT_DESC)

## 12. Actual vs predicted plots
*Paper analogue: Fig. 10.* Measured (open circles) and predicted (filled) values against the sample index, with the
training, validation and test samples shown one after another (within each set samples follow the lab-file order:
IR → temperature → time).

In [ ]:
with step("12. Actual vs predicted plots"):
    fig, axs = model_grid(len(MODELS_OK), w=4.6, h=2.7)
    for ax, m in zip(axs, MODELS_OK):
        off = 0
        for s, ix in SETS.items():
            xi = np.arange(off, off + len(ix)); off += len(ix)
            ax.scatter(xi, y_all[ix], s=13, facecolors="none", edgecolors=SET_COLORS[s], lw=0.6, alpha=0.7)
            ax.scatter(xi, PREDS[m][s], s=7, color=SET_COLORS[s], alpha=0.9)
        ax.set_title(f"{m} - actual vs predicted", fontsize=8.5); ax.set_xlabel("Data index"); ax.set_ylabel(f"{short(TARGET)} ({UNIT})")
    handles = []
    for s in SETS:
        handles += [Line2D([], [], marker="o", ls="", mfc="none", mec=SET_COLORS[s], label=f"Actual {s.lower()}"),
                    Line2D([], [], marker="o", ls="", color=SET_COLORS[s], label=f"{s} predictions")]
    fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 0.0))
    finish_grid(fig, f"Actual vs predicted {short(TARGET).lower()} by data index ({SPLIT_DESC})", bottom=0.025)
    save_fig(fig, "F13_actual_vs_predicted_index", "prediction", "Actual vs predicted by index",
             "Measured and predicted values for train, validation and test samples of every model.", "Fig. 10", len(y_all), SPLIT_DESC)

## 13. Cross-validation plots
*Paper analogue: Fig. 11 (cross plots, called "cross-validation graphs" in the paper's text).* Predicted vs measured with
the y = x bisector for the train / validation / test split. Two further cross-plot grids show the genuine **out-of-fold**
predictions of the leave-one-temperature-out and leave-one-time-out cross-validation (Section 9b).

In [ ]:
def cross_plot_grid(get_xy, title, key, paper, validation, n_note):
    fig, axs = model_grid(len(MODELS_OK), w=3.6, h=3.4)
    lo, hi = y_all.min(), y_all.max(); pad = 0.05 * (hi - lo); lim = (lo - pad, hi + pad)
    for ax, m in zip(axs, MODELS_OK):
        parts = get_xy(m); ttl = []
        for s, (yt, yp) in parts.items():
            ax.scatter(yt, yp, s=10, alpha=0.65, color=SET_COLORS.get(s, "#6b46c1"), label=s, edgecolor="none")
            ttl.append(f"{s[:5] if s in SET_COLORS else s} R²={metrics(yt, yp)['R2']:.2f}")
        ax.plot(lim, lim, "k--", lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
        ax.set_title(f"{m}\n" + ", ".join(ttl), fontsize=7.5)
        ax.set_xlabel(f"Actual ({UNIT})", fontsize=7.5); ax.set_ylabel(f"Predicted ({UNIT})", fontsize=7.5)
    axs[0].legend(fontsize=6.5, loc="upper left")
    finish_grid(fig, title)
    save_fig(fig, key, "prediction", title.split(" (")[0], "Predicted vs measured with the y = x bisector.", paper, n_note, validation)

with step("13. Cross-validation plots"):
    cross_plot_grid(lambda m: {s: (y_all[ix], PREDS[m][s]) for s, ix in SETS.items()},
                    f"Cross plots: predicted vs actual ({SPLIT_DESC})", "F14_cross_plots", "Fig. 11", SPLIT_DESC, len(y_all))
    if RUN_GROUPED_BENCHMARK and "GOOF" in globals() and GOOF is not None:
        for s in GOOF.scheme.unique():
            g = GOOF[GOOF.scheme == s]
            cross_plot_grid(lambda m, g=g: {"out-of-fold": (g[g.model == m].actual.values, g[g.model == m].predicted.values)},
                            f"{SCHEME_NAME[s]} cross-validation (out-of-fold, n = {g.row_id.nunique()})",
                            f"F15_cv_cross_plots_{s}", "Fig. 11 (grouped CV variant)", SCHEME_NAME[s] + ", nested grouped tuning",
                            g.row_id.nunique())

## 14. Relative deviation plots
*Paper analogue: Fig. 12.* Absolute relative deviation 100·|y − ŷ|/y for every sample. Note that low measured values
(e.g. IR = 0, surface areas of a few tens) inflate relative deviations even when absolute errors are small.

In [ ]:
with step("14. Relative deviation plots"):
    fig, axs = model_grid(len(MODELS_OK), w=4.4, h=2.7)
    RD_ROWS = []
    for ax, m in zip(axs, MODELS_OK):
        for s, ix in SETS.items():
            rd = 100 * np.abs(y_all[ix] - PREDS[m][s]) / np.abs(y_all[ix])
            ax.plot(np.arange(len(ix)), rd, color=SET_COLORS[s], lw=0.9, label=s)
            RD_ROWS.append(dict(model=m, set=s, mean_RD=rd.mean(), median_RD=np.median(rd), max_RD=rd.max()))
        ax.set_title(f"{m} relative deviation %", fontsize=8.5); ax.set_xlabel("Index within set"); ax.set_ylabel("Relative deviation (%)")
    axs[0].legend(fontsize=6.5)
    finish_grid(fig, f"Absolute relative deviation by sample ({SPLIT_DESC})")
    save_table(pd.DataFrame(RD_ROWS), "relative_deviation_summary", index=False)
    save_fig(fig, "F16_relative_deviation", "prediction", "Relative deviation", "Absolute relative deviation (%) per sample, per set, per model.",
             "Fig. 12", len(y_all), SPLIT_DESC)

## 15. Prediction distribution histograms
*Paper analogue: Fig. 13.* Histograms of predicted values for each set; the black outline is the distribution of the
measured values of all sets for comparison.

In [ ]:
with step("15. Prediction distribution histograms"):
    fig, axs = model_grid(len(MODELS_OK), w=4.2, h=2.8)
    bins = np.linspace(min(y_all.min(), min(PREDS[m][s].min() for m in MODELS_OK for s in SETS)),
                       max(y_all.max(), max(PREDS[m][s].max() for m in MODELS_OK for s in SETS)), 26)
    for ax, m in zip(axs, MODELS_OK):
        for s in SETS:
            ax.hist(PREDS[m][s], bins=bins, alpha=0.55, color=SET_COLORS[s], label=f"{s} predictions")
        ax.hist(y_all, bins=bins, histtype="step", color="k", lw=0.9, label="Measured (all)")
        ax.set_title(f"{m} prediction distributions", fontsize=8.5); ax.set_xlabel(f"Predicted ({UNIT})"); ax.set_ylabel("Frequency")
    axs[0].legend(fontsize=6.5)
    finish_grid(fig, f"Distribution of predicted values ({SPLIT_DESC})")
    save_fig(fig, "F17_prediction_distributions", "prediction", "Prediction distributions", "Histograms of predicted values per set.",
             "Fig. 13", len(y_all), SPLIT_DESC)

## 16. Feature importance and SHAP analysis
*Paper analogue: Fig. 14 (A: Random-Forest importance, B: mean SHAP).* Shown for the Random Forest, the best model by
validation R² and the Gaussian process. Tree models use the exact TreeExplainer; the others use KernelExplainer (exact
enough for 2-3 inputs). Permutation importance on the **test** set is added as a model-agnostic check.

In [ ]:
def shap_values_for(name, est, X_bg, X_explain):
    base = base_estimator(est)
    if name in TREE_MODELS:
        try:
            return np.asarray(shap.TreeExplainer(base).shap_values(X_explain))
        except Exception as e:
            print(f"  TreeExplainer failed for {name} ({e}); using KernelExplainer")
    ex = shap.KernelExplainer(est.predict, shap.kmeans(X_bg, min(20, len(X_bg))))
    return np.asarray(ex.shap_values(X_explain, nsamples=300, silent=True))

with step("16. Feature importance and SHAP analysis"):
    explain = [m for m in dict.fromkeys(["Random Forest", BEST, "Gaussian Process"]) if m in FITTED]
    SHAP = {}
    for m in tqdm(explain, desc="SHAP"):
        SHAP[m] = shap_values_for(m, FITTED[m], X_all[SETS["Train"]], X_all)
    fnames = [short(f) for f in FEATURES]

    # Fig: RF impurity importance (A) + mean |SHAP| (B) as in the paper
    rf = base_estimator(FITTED["Random Forest"]) if "Random Forest" in FITTED else None
    fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.4))
    if rf is not None:
        imp = pd.Series(rf.feature_importances_, index=fnames).sort_values()
        axs[0].barh(imp.index, imp.values, color=sns.color_palette("viridis", len(imp))); axs[0].set_xlabel("Importance score (mean impurity decrease)")
        axs[0].set_title("A  Feature importance from Random Forest")
    msh = pd.Series(np.abs(SHAP[BEST]).mean(0), index=fnames).sort_values()
    axs[1].barh(msh.index, msh.values, color=sns.color_palette("plasma", len(msh))); axs[1].set_xlabel(f"Mean |SHAP value| ({UNIT})")
    axs[1].set_title(f"B  Mean SHAP values - {BEST}")
    fig.tight_layout()
    save_fig(fig, "F18_rf_importance_and_shap", "explainability", "Random-Forest importance and mean SHAP",
             "A: impurity-based RF importance. B: mean absolute SHAP value of the best model.", "Fig. 14", len(y_all),
             f"models trained on the training set; SHAP evaluated on all {len(y_all)} samples")

    # Fig: SHAP beeswarm for each explained model
    fig, axs = plt.subplots(1, len(explain), figsize=(4.8 * len(explain), 3.2), squeeze=False)
    for ax, m in zip(axs.ravel(), explain):
        plt.sca(ax)
        shap.summary_plot(SHAP[m], X_all, feature_names=fnames, show=False, plot_size=None, color_bar=(m == explain[-1]))
        ax.set_title(m, fontsize=9); ax.set_xlabel(f"SHAP value ({UNIT})")
    fig.tight_layout()
    save_fig(fig, "F19_shap_beeswarm", "explainability", "SHAP summary (beeswarm)", "Per-sample SHAP values coloured by feature value.",
             "Fig. 14B (extended)", len(y_all), "SHAP on all samples")

    # Fig: SHAP dependence for the best model
    fig, axs = plt.subplots(1, len(FEATURES), figsize=(4 * len(FEATURES), 3.2))
    axs = np.atleast_1d(axs)
    for j, (ax, f) in enumerate(zip(axs, FEATURES)):
        other = FEATURES[(j + 1) % len(FEATURES)]
        scat = ax.scatter(X_all[:, j], SHAP[BEST][:, j], c=X_all[:, FEATURES.index(other)], cmap="coolwarm", s=12)
        ax.set_xlabel(label(f)); ax.set_ylabel(f"SHAP for {short(f)} ({UNIT})"); fig.colorbar(scat, ax=ax, label=short(other))
    fig.suptitle(f"SHAP dependence - {BEST}"); fig.tight_layout()
    save_fig(fig, "F20_shap_dependence", "explainability", "SHAP dependence", "SHAP value vs feature value, coloured by a second feature.",
             "—", len(y_all), "SHAP on all samples")

    # table: all importance measures
    imp_rows = []
    for m in explain:
        for j, f in enumerate(FEATURES):
            imp_rows.append(dict(model=m, feature=f, mean_abs_shap=float(np.abs(SHAP[m][:, j]).mean())))
    for m in explain:
        pi = permutation_importance(FITTED[m], X_all[SETS["Test"]], y_all[SETS["Test"]], n_repeats=30, random_state=SEED,
                                    scoring="neg_mean_squared_error")
        for j, f in enumerate(FEATURES):
            imp_rows.append(dict(model=m, feature=f, permutation_dMSE_test=float(pi.importances_mean[j])))
    IMP = pd.DataFrame(imp_rows).groupby(["model", "feature"]).first().reset_index()
    if rf is not None:
        IMP = IMP.merge(pd.DataFrame(dict(model="Random Forest", feature=FEATURES, rf_impurity=rf.feature_importances_)), how="left")
    display(IMP.round(3)); save_table(IMP, "feature_importance", index=False)
    if SAVED_RESULTS is not None and (SAVED_RESULTS / "shap_mean_abs_SA.csv").exists() and TARGET == "Sur_Area":
        print("Saved repository SHAP (GP trained on ALL data, results/shap_mean_abs_SA.csv):")
        display(pd.read_csv(SAVED_RESULTS / "shap_mean_abs_SA.csv", index_col=0).round(1).T)

## 17. Gaussian Process uncertainty analysis  *(project-specific)*
The project model is the Gaussian process (surface area: Matérn-5/2 ARD GP; yield: linear trend + GP, as in
`03_final_models.py`). Here it is evaluated with **leave-one-temperature-out** and **leave-one-time-out** validation,
including its predictive standard deviation.

**Prediction-interval calibration.** Raw GP 95 % intervals can be too narrow at unseen conditions. A multiplier *k* is
chosen so that 95 % of held-out |z| = |y − μ|/σ fall below it. To report an *honest* coverage, *k* is **cross-fitted**:
for every held-out level it is computed from the other levels only and then tested on that level. The global *k* (all
levels) is what the final model uses. Because a single multiplier is tuned for the 95 % level, narrower intervals
(e.g. 50-80 %) become conservative (over-cover); the reliability diagram shows this explicitly.

In [ ]:
def make_project_gp():
    return make_gp_pipeline(len(FEATURES)) if TARGET == "Sur_Area" else TrendGP(n_feat=len(FEATURES))

with step("17. Gaussian Process uncertainty analysis"):
    recs = []
    folds = list(grouped_splits(data_model))
    for scheme, fold, tr, te, grp in tqdm(folds, desc="GP grouped CV"):
        mgp = make_project_gp().fit(X_all[tr], y_all[tr]); mu, sd = mgp.predict(X_all[te], return_std=True)
        recs.append(pd.DataFrame(dict(scheme=scheme, fold=fold, row_id=data_model.row_id.values[te],
                                      **{f: X_all[te, j] for j, f in enumerate(FEATURES)}, y=y_all[te], mu=mu, sd=sd)))
    GPCV = pd.concat(recs, ignore_index=True); GPCV["absz"] = np.abs(GPCV.y - GPCV.mu) / GPCV.sd
    K95 = max(1.96, float(np.quantile(GPCV.absz, 0.95)))
    GPCV["k_crossfit"] = np.nan
    for f in GPCV.fold.unique():
        msk = GPCV.fold == f
        GPCV.loc[msk, "k_crossfit"] = max(1.96, float(np.quantile(GPCV.absz[~msk], 0.95)))
    save_table(GPCV, "gp_grouped_cv_with_uncertainty", index=False)
    rows = []
    for s, g in GPCV.groupby("scheme"):
        rows.append(dict(scheme=SCHEME_NAME[s], n=len(g), **{k: v for k, v in metrics(g.y, g.mu).items() if k != "n"},
                         coverage95_raw=(g.absz <= 1.96).mean(), coverage68_raw=(g.absz <= 1.0).mean(),
                         coverage95_crossfit=(g.absz <= g.k_crossfit).mean()))
    rows.append(dict(scheme="All held-out levels", n=len(GPCV), **{k: v for k, v in metrics(GPCV.y, GPCV.mu).items() if k != "n"},
                     coverage95_raw=(GPCV.absz <= 1.96).mean(), coverage68_raw=(GPCV.absz <= 1).mean(),
                     coverage95_crossfit=(GPCV.absz <= GPCV.k_crossfit).mean()))
    GPSUM = pd.DataFrame(rows).set_index("scheme"); display(GPSUM.round(3)); save_table(GPSUM, "gp_uncertainty_summary")
    print(f"Global interval multiplier k95 = {K95:.2f} (raw Gaussian value 1.96). "
          f"Cross-fitted 95 % coverage = {GPSUM.loc['All held-out levels', 'coverage95_crossfit']:.1%} "
          "(honest estimate; the in-sample calibrated coverage is 95 % by construction).")
    saved_cv = {"Sur_Area": "cv_SA_gp_with_uncertainty.csv", "Yield": "cv_Yield_IR4.csv"}[TARGET]
    if SAVED_RESULTS is not None and (SAVED_RESULTS / saved_cv).exists():
        prev = pd.read_csv(SAVED_RESULTS / saved_cv)
        prevm = pd.DataFrame([dict(scheme=SCHEME_NAME[s], R2_saved=metrics(g.y, g.mu)["R2"], RMSE_saved=metrics(g.y, g.mu)["RMSE"],
                                   coverage95_saved=(np.abs(g.y - g.mu) / g.sd <= 1.96).mean()) for s, g in prev.groupby("scheme")]).set_index("scheme")
        display(Markdown(f"Saved repository predictions (`results/{saved_cv}`) for comparison:")); display(prevm.round(3))

    # Fig: parity with calibrated intervals
    schemes = list(GPCV.scheme.unique())
    fig, axs = plt.subplots(1, len(schemes), figsize=(5.2 * len(schemes), 4.6), squeeze=False)
    lo, hi = y_all.min(), y_all.max(); pad = 0.06 * (hi - lo); lim = (lo - pad, hi + pad)
    for ax, s in zip(axs.ravel(), schemes):
        g = GPCV[GPCV.scheme == s]
        ax.errorbar(g.y, g.mu, yerr=K95 * g.sd, fmt="o", ms=3.2, alpha=0.6, elinewidth=0.6, color="#1f4e79", ecolor="#7aa6d6")
        ax.plot(lim, lim, "k--", lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
        mm = metrics(g.y, g.mu)
        ax.set_title(f"{SCHEME_NAME[s]} (n = {len(g)})\nR² = {mm['R2']:.3f}, RMSE = {mm['RMSE']:.1f} {UNIT}; "
                     f"95 % coverage raw {np.mean(g.absz <= 1.96):.0%}, cross-fitted {np.mean(g.absz <= g.k_crossfit):.0%}", fontsize=8)
        ax.set_xlabel(f"Measured {short(TARGET).lower()} ({UNIT})"); ax.set_ylabel(f"GP prediction ± {K95:.2f}σ ({UNIT})")
    fig.tight_layout()
    save_fig(fig, "F21_gp_parity_intervals", "uncertainty", "GP parity with calibrated 95 % intervals",
             "Out-of-fold GP predictions at unseen temperatures / times with calibrated error bars.", "— (project)", len(GPCV),
             "leave-one-temperature-out and leave-one-time-out")

    # Fig: calibration (reliability diagram + z histogram)
    lv = np.linspace(0.5, 0.99, 15); qz = stats.norm.ppf((1 + lv) / 2)
    raw_cov = [(GPCV.absz <= q).mean() for q in qz]
    cal_cov = [(GPCV.absz / (GPCV.k_crossfit / 1.96) <= q).mean() for q in qz]
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].plot([0.5, 1], [0.5, 1], "k--", lw=0.8, label="Perfect calibration")
    axs[0].plot(lv, raw_cov, "o-", color="#c53030", ms=4, label="Raw GP intervals")
    axs[0].plot(lv, cal_cov, "s-", color="#2f855a", ms=4, label="Calibrated (cross-fitted k)")
    axs[0].set_xlabel("Nominal coverage"); axs[0].set_ylabel("Observed coverage (held-out)"); axs[0].legend()
    axs[0].set_title(f"Reliability diagram (n = {len(GPCV)} held-out predictions)")
    zz = (GPCV.y - GPCV.mu) / GPCV.sd
    axs[1].hist(zz, bins=30, density=True, color="#bee3f8", edgecolor="#2c5282", label="Standardised errors z")
    xs = np.linspace(-6, 6, 300)
    axs[1].plot(xs, stats.norm.pdf(xs), "r-", label="N(0, 1) (raw GP assumption)")
    axs[1].plot(xs, stats.norm.pdf(xs, scale=K95 / 1.96), "g--", label=f"N(0, {K95 / 1.96:.2f}²) after calibration")
    axs[1].set_xlabel("z = (measured − predicted) / σ"); axs[1].set_ylabel("Density"); axs[1].legend(fontsize=7)
    axs[1].set_title(f"k95 = {K95:.2f}")
    fig.tight_layout()
    save_fig(fig, "F22_interval_calibration", "uncertainty", "Prediction-interval calibration",
             "Reliability diagram of raw and cross-fitted calibrated intervals, and distribution of standardised errors.",
             "— (project)", len(GPCV), "grouped CV; cross-fitted calibration")

    # ---- final GP on all modelling data ----
    FINAL_GP = make_project_gp().fit(X_all, y_all)
    sc_, gp_, trend_ = gp_parts(FINAL_GP)
    joblib.dump(dict(model=FINAL_GP, k95=K95, features=FEATURES, target=TARGET, unit=UNIT, data_hash=DATA_HASH,
                     domain={f: [float(data_model[f].min()), float(data_model[f].max())] for f in FEATURES},
                     created=f"{datetime.datetime.now():%Y-%m-%d %H:%M}"), DIRS["models"] / f"{TARGET}_gp_final.joblib")
    print("Final GP kernel:", gp_.kernel_)
    if TARGET == "Sur_Area":
        ls = gp_.kernel_.k1.k2.length_scale
        print("ARD length-scales (standardised units; shorter = more influential):", dict(zip(FEATURES, np.round(ls, 3))))

    # closed-form leave-one-out residuals of the final GP (fixed hyper-parameters; Rasmussen & Williams eq. 5.12)
    Z = sc_.transform(X_all); target_vec = y_all - (trend_.predict(Z) if trend_ is not None else 0)
    Ki = np.linalg.inv(gp_.kernel_(Z) + gp_.alpha * np.eye(len(Z)))
    yn = (target_vec - gp_._y_train_mean) / gp_._y_train_std
    loo_mu = (yn - (Ki @ yn) / np.diag(Ki)) * gp_._y_train_std + gp_._y_train_mean
    loo_sd = np.sqrt(1 / np.diag(Ki)) * gp_._y_train_std
    data_model["gp_loo_z"] = (target_vec - loo_mu) / loo_sd
    LOOF = data_model.sort_values("gp_loo_z", key=np.abs, ascending=False).head(12)
    display(Markdown("**Largest leave-one-out standardised residuals of the final GP** (check against lab notebooks)"))
    display(LOOF[["row_id"] + FEATURES + [TARGET, "gp_loo_z", "mc_z"]].round(2)); save_table(LOOF, "gp_loo_quality_flags", index=False)

    # Fig: GP uncertainty maps
    Tg, tg = np.linspace(800, 1200, 81), np.linspace(0, 90, 91); TT, tt = np.meshgrid(Tg, tg)
    def make_X(**kw):
        arrs = np.broadcast_arrays(*[np.asarray(kw[f], float) for f in FEATURES]); return np.column_stack([a.ravel() for a in arrs])
    panels = IR_LEVELS if "IR" in FEATURES else [4]
    ncol = 4 if len(panels) > 1 else 1; nrow = int(np.ceil(len(panels) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.7 * ncol, 3.1 * nrow), squeeze=False, constrained_layout=True)
    hw_all = {ir: K95 * FINAL_GP.predict(make_X(IR=ir, Act_Temp=TT, Act_Time=tt), return_std=True)[1].reshape(TT.shape) for ir in panels}
    vmax_u = max(v.max() for v in hw_all.values())
    for ax, ir in zip(axs.ravel(), panels):
        cs = ax.contourf(TT, tt, hw_all[ir], levels=np.linspace(0, vmax_u, 16), cmap="Blues")
        d = data_model[data_model.IR == ir]; ax.scatter(d.Act_Temp, d.Act_Time, c="k", s=6)
        ax.set_title(f"IR = {ir}"); ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)"); ax.grid(False)
    for ax in axs.ravel()[len(panels):]:
        ax.axis("off")
    fig.colorbar(cs, ax=axs, shrink=0.75, label=f"Calibrated 95 % half-width ({UNIT})")
    fig.suptitle(f"GP uncertainty maps (final GP on n = {len(y_all)}; dots = measured conditions)")
    save_fig(fig, "F23_gp_uncertainty_maps", "uncertainty", "GP uncertainty maps",
             "Calibrated 95 % interval half-width over temperature × time for each IR.", "— (project)", len(y_all),
             "final GP on all data; k95 from grouped CV")

## 18. Response surface visualizations  *(project-specific)*
GP mean predictions over the measured domain (no extrapolation outside 800-1200 °C, 0-90 min, IR 0-6), with measured
conditions overlaid, response curves with calibrated bands, a 3-D surface, and a comparison with tree models (which give
step-shaped surfaces between grid points).

In [ ]:
with step("18. Response surface visualizations"):
    vmax = float(y_all.max()) * 1.05
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.7 * ncol, 3.1 * nrow), squeeze=False, constrained_layout=True)
    for ax, ir in zip(axs.ravel(), panels):
        mu = FINAL_GP.predict(make_X(IR=ir, Act_Temp=TT, Act_Time=tt)).reshape(TT.shape)
        cs = ax.contourf(TT, tt, mu, levels=np.linspace(min(0, y_all.min()), vmax, 21), cmap="viridis", extend="both")
        ax.contour(TT, tt, mu, levels=8, colors="w", linewidths=0.3, alpha=0.6)
        d = data_model[data_model.IR == ir]
        ax.scatter(d.Act_Temp, d.Act_Time, c="white", s=10, edgecolor="k", lw=0.4)
        k = np.unravel_index(np.argmax(mu), mu.shape); ax.plot(TT[k], tt[k], "r*", ms=9)
        ax.set_title(f"IR = {ir}  (max {mu.max():.0f} at {TT[k]:.0f} °C, {tt[k]:.0f} min)", fontsize=8)
        ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)"); ax.grid(False)
    for ax in axs.ravel()[len(panels):]:
        ax.axis("off")
    fig.colorbar(cs, ax=axs, shrink=0.75, label=f"Predicted {label(TARGET)[0].lower()}{label(TARGET)[1:]}")
    fig.suptitle(f"GP response surfaces (red star = predicted maximum; circles = measured, n = {len(y_all)})")
    save_fig(fig, "F24_response_surfaces", "uncertainty", "Response surfaces", "GP mean surface over temperature × time for each IR.",
             "— (project)", len(y_all), "final GP on all data")

    # curves with calibrated bands
    curve_irs = [ir for ir in panels if ir >= 3] or panels
    cols_ = plt.cm.plasma(np.linspace(0.05, 0.85, len(curve_irs)))
    t_fix, T_fix = 60, 1000
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.1))
    for ir, c in zip(curve_irs, cols_):
        Tq = np.linspace(800, 1200, 161); mu, sd = FINAL_GP.predict(make_X(IR=ir, Act_Temp=Tq, Act_Time=t_fix), return_std=True)
        axs[0].plot(Tq, mu, color=c, label=f"IR {ir}"); axs[0].fill_between(Tq, mu - K95 * sd, mu + K95 * sd, color=c, alpha=0.15)
        d = data_model[(data_model.IR == ir) & (data_model.Act_Time == t_fix)]; axs[0].scatter(d.Act_Temp, d[TARGET], color=c, s=22, edgecolor="k", lw=0.5, zorder=3)
        tq = np.linspace(0, 90, 181); mu, sd = FINAL_GP.predict(make_X(IR=ir, Act_Temp=T_fix, Act_Time=tq), return_std=True)
        axs[1].plot(tq, mu, color=c); axs[1].fill_between(tq, mu - K95 * sd, mu + K95 * sd, color=c, alpha=0.15)
        d = data_model[(data_model.IR == ir) & (data_model.Act_Temp == T_fix)]; axs[1].scatter(d.Act_Time, d[TARGET], color=c, s=22, edgecolor="k", lw=0.5, zorder=3)
    axs[0].set_xlabel(f"{label('Act_Temp')}  [time = {t_fix} min]"); axs[1].set_xlabel(f"{label('Act_Time')}  [temperature = {T_fix} °C]")
    for a in axs:
        a.set_ylabel(label(TARGET))
    axs[0].legend(ncol=2); fig.suptitle(f"GP response curves with calibrated 95 % bands (k = {K95:.2f}); points = measurements"); fig.tight_layout()
    save_fig(fig, "F25_response_curves", "uncertainty", "Response curves with intervals", "GP mean and calibrated band vs temperature and vs time.",
             "— (project)", len(y_all), "final GP; k95 from grouped CV")

    # 3-D surface at the IR with the highest measured value
    ir_best = int(data_model.loc[data_model[TARGET].idxmax(), "IR"])
    mu = FINAL_GP.predict(make_X(IR=ir_best, Act_Temp=TT, Act_Time=tt)).reshape(TT.shape)
    fig = plt.figure(figsize=(7, 5.2)); ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(TT, tt, mu, cmap="viridis", alpha=0.85, linewidth=0, antialiased=True)
    d = data_model[data_model.IR == ir_best]; ax.scatter(d.Act_Temp, d.Act_Time, d[TARGET], c="r", s=12, depthshade=False)
    ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)"); ax.set_zlabel(label(TARGET)); ax.view_init(28, -130)
    ax.set_title(f"GP surface at IR = {ir_best} (red = measured, n = {len(d)})")
    save_fig(fig, "F26_response_surface_3d", "uncertainty", "3-D response surface", "3-D GP surface with measured points.", "— (project)",
             len(d), "final GP on all data")

    # comparison GP vs tree models refitted on all data with their tuned settings
    comp_models = [m for m in ("XGBoost", "Random Forest") if m in FITTED]
    fig, axs = plt.subplots(1, 1 + len(comp_models), figsize=(4.2 * (1 + len(comp_models)), 3.6), squeeze=False, constrained_layout=True)
    surf = {"Gaussian Process (project)": mu}
    for m in comp_models:
        mdl = clone(base_estimator(FITTED[m])).fit(X_all, y_all)
        surf[m] = mdl.predict(make_X(IR=ir_best, Act_Temp=TT, Act_Time=tt)).reshape(TT.shape)
    for ax, (m, s_) in zip(axs.ravel(), surf.items()):
        cs = ax.contourf(TT, tt, s_, levels=np.linspace(min(0, y_all.min()), vmax, 21), cmap="viridis", extend="both")
        ax.scatter(d.Act_Temp, d.Act_Time, c="white", s=10, edgecolor="k", lw=0.4); ax.set_title(m); ax.grid(False)
        ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)")
    fig.colorbar(cs, ax=axs, shrink=0.8, label=label(TARGET))
    fig.suptitle(f"Response surface at IR = {ir_best}: GP vs tree models (all fitted on n = {len(y_all)})")
    save_fig(fig, "F27_surface_model_comparison", "uncertainty", "Surface comparison GP vs trees",
             "Same-condition surfaces from the GP and tree ensembles (tuned settings, refitted on all data).", "— (project)",
             len(y_all), "all models refitted on all data")

## 19. Active learning and experimental design  *(project-specific)*
Each real sample costs two days, so the GP proposes where to measure next:
* **Exploit - expected improvement (EI)** over the best measured value (find a higher target value).
* **Explore - greedy maximum-variance batch** (reduce model uncertainty most; exact for a GP because variance does not
  depend on the unmeasured outcomes).

Candidates: integer IR within the measured range, 800-1200 °C in 25 °C steps, 0-90 min in 5 min steps, excluding conditions
already measured. Predictions are shown with calibrated 95 % intervals. A **retrospective simulation** on the measured
data shows how quickly each strategy would have learned the surface or found the maximum (it re-uses only real
measurements; the GP hyper-parameters are fixed at their full-data values, so it is slightly optimistic).

In [ ]:
with step("19. Active learning and experimental design"):
    Tc, tc = np.arange(800, 1201, 25), np.arange(0, 91, 5)
    irs_c = panels
    cand = pd.DataFrame([(a, b, c) for a in irs_c for b in Tc for c in tc], columns=["IR", "Act_Temp", "Act_Time"])
    measured = set(map(tuple, data_model[["IR", "Act_Temp", "Act_Time"]].astype(float).values))
    cand = cand[[tuple(map(float, r)) not in measured for r in cand.values]].reset_index(drop=True)
    Xc = cand[FEATURES].to_numpy(float)
    mu_c, sd_c = FINAL_GP.predict(Xc, return_std=True)
    best_y = y_all.max(); zc = (mu_c - best_y) / sd_c
    cand["pred"], cand["sd"] = mu_c, sd_c
    cand["lo95"], cand["hi95"] = mu_c - K95 * sd_c, mu_c + K95 * sd_c
    cand["EI"] = sd_c * (zc * stats.norm.cdf(zc) + stats.norm.pdf(zc))
    EXPLOIT = cand.sort_values("EI", ascending=False).head(10).reset_index(drop=True)

    kc = gp_.kernel_.k1; noise = gp_.kernel_.k2.noise_level
    Zc, Zt = sc_.transform(Xc), sc_.transform(X_all)
    def post_var(Ztrain, Zq):
        K = kc(Ztrain) + noise * np.eye(len(Ztrain)); Ks = kc(Zq, Ztrain)
        return np.diag(kc(Zq)) - np.einsum("ij,ij->i", Ks @ np.linalg.inv(K), Ks)
    sel, Zcur = [], Zt.copy()
    for _ in tqdm(range(10), desc="Greedy max-variance batch"):
        v = post_var(Zcur, Zc); v[sel] = -np.inf; j = int(np.argmax(v)); sel.append(j); Zcur = np.vstack([Zcur, Zc[j]])
    EXPLORE = cand.loc[sel].reset_index(drop=True); EXPLORE.insert(0, "order", range(1, len(sel) + 1))
    cols_show = ["IR", "Act_Temp", "Act_Time", "pred", "lo95", "hi95", "EI"]
    display(Markdown(f"**Exploit - top-10 expected improvement over the best measured value {best_y:.0f} {UNIT}** "
                     f"(intervals: calibrated 95 %, k = {K95:.2f})")); display(EXPLOIT[cols_show].round(1))
    display(Markdown("**Explore - greedy maximum-information batch of 10**")); display(EXPLORE[["order"] + cols_show[:-1]].round(1))
    save_table(EXPLOIT, "next_experiments_exploit_EI", index=False); save_table(EXPLORE, "next_experiments_explore", index=False)

    # Fig: EI maps for the IR levels with the highest EI + top candidates
    top_irs = cand.groupby("IR").EI.max().sort_values(ascending=False).index[:3].tolist() if "IR" in FEATURES else panels
    fig, axs = plt.subplots(1, len(top_irs), figsize=(4.3 * len(top_irs), 3.6), squeeze=False, constrained_layout=True)
    eimax = cand.EI.max()
    for ax, ir in zip(axs.ravel(), top_irs):
        ei = cand[cand.IR == ir].pivot_table(index="Act_Time", columns="Act_Temp", values="EI")
        cs = ax.contourf(ei.columns, ei.index, ei.values, levels=np.linspace(0, eimax, 16), cmap="magma")
        d = data_model[data_model.IR == ir]; ax.scatter(d.Act_Temp, d.Act_Time, c="white", s=8, edgecolor="k", lw=0.3)
        e = EXPLOIT[EXPLOIT.IR == ir]
        ax.scatter(e.Act_Temp, e.Act_Time, marker="*", c="cyan", s=90, edgecolor="k", lw=0.5)
        ax.set_title(f"IR = {ir}"); ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)"); ax.grid(False)
    fig.colorbar(cs, ax=axs, shrink=0.8, label=f"Expected improvement ({UNIT})")
    fig.suptitle(f"Expected improvement over the best measurement ({best_y:.0f} {UNIT}); stars = top-10 candidates")
    save_fig(fig, "F28_expected_improvement_maps", "experimental_design", "Expected-improvement maps",
             "EI of unmeasured candidate conditions with the top-10 suggestions.", "— (project)", len(y_all), "final GP on all data")

    # Fig: exploration batch on uncertainty maps
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.7 * ncol, 3.1 * nrow), squeeze=False, constrained_layout=True)
    for ax, ir in zip(axs.ravel(), panels):
        cs = ax.contourf(TT, tt, hw_all[ir], levels=np.linspace(0, vmax_u, 16), cmap="Blues")
        d = data_model[data_model.IR == ir]; ax.scatter(d.Act_Temp, d.Act_Time, c="k", s=5)
        e = EXPLORE[EXPLORE.IR == ir]
        for _, r in e.iterrows():
            ax.scatter(r.Act_Temp, r.Act_Time, marker="o", s=80, facecolor="#f6ad55", edgecolor="k", zorder=4)
            ax.annotate(str(int(r.order)), (r.Act_Temp, r.Act_Time), ha="center", va="center", fontsize=6.5, zorder=5)
        ax.set_title(f"IR = {ir}"); ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Time (min)"); ax.grid(False)
    for ax in axs.ravel()[len(panels):]:
        ax.axis("off")
    fig.colorbar(cs, ax=axs, shrink=0.75, label=f"Calibrated 95 % half-width ({UNIT})")
    fig.suptitle("Maximum-information batch (numbered in selection order) on the current uncertainty maps")
    save_fig(fig, "F29_exploration_batch", "experimental_design", "Exploration batch",
             "Greedy maximum-variance batch of 10 experiments over the uncertainty maps.", "— (project)", len(y_all), "final GP on all data")

    # retrospective active-learning simulation on measured data only
    n0, n_add, reps = (10, 50, 10) if len(y_all) > 100 else (5, 20, 10)
    Zs = REF_GP[0].transform(X_all)
    def simulate(strategy, seed):
        r = np.random.default_rng(seed); n = len(y_all); chosen = list(r.choice(n, n0, replace=False)); out = []
        for k in range(n_add + 1):
            rest = np.setdiff1d(np.arange(n), chosen)
            g = GaussianProcessRegressor(REF_KERNEL, optimizer=None, normalize_y=True).fit(Zs[chosen], y_all[chosen])
            mu, sd = g.predict(Zs[rest], return_std=True)
            out.append((len(chosen), np.sqrt(np.mean((y_all[rest] - mu) ** 2)), y_all[chosen].max()))
            if k == n_add:
                break
            if strategy == "Random":
                j = r.choice(rest)
            elif strategy == "Max variance (explore)":
                j = rest[np.argmax(sd)]
            else:
                b = y_all[chosen].max(); z = (mu - b) / np.maximum(sd, 1e-9); j = rest[np.argmax(sd * (z * stats.norm.cdf(z) + stats.norm.pdf(z)))]
            chosen.append(int(j))
        return out
    sim = []
    strategies = ["Random", "Max variance (explore)", "Expected improvement (exploit)"]
    for st in strategies:
        for rep in tqdm(range(reps), desc=st):
            for nn_, rm, bb in simulate(st, SEED + rep):
                sim.append(dict(strategy=st, rep=rep, n_measured=nn_, rmse_rest=rm, best_found=bb))
    SIM = pd.DataFrame(sim); save_table(SIM, "active_learning_simulation", index=False)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    colors_s = dict(zip(strategies, ["#718096", "#2b6cb0", "#c53030"]))
    for st in strategies:
        g = SIM[SIM.strategy == st].groupby("n_measured")
        for ax, col in zip(axs, ["rmse_rest", "best_found"]):
            m_, s_ = g[col].mean(), g[col].std()
            ax.plot(m_.index, m_.values, color=colors_s[st], label=st); ax.fill_between(m_.index, m_ - s_, m_ + s_, color=colors_s[st], alpha=0.15)
    axs[0].set_ylabel(f"RMSE on not-yet-measured samples ({UNIT})"); axs[1].set_ylabel(f"Best {short(TARGET).lower()} measured so far ({UNIT})")
    axs[1].axhline(y_all.max(), color="k", ls=":", lw=0.8, label="Best in full dataset")
    for ax in axs:
        ax.set_xlabel("Number of experiments"); sec = ax.secondary_xaxis("top", functions=(lambda x: 2 * x, lambda x: x / 2)); sec.set_xlabel("Lab days (2 per sample)")
    axs[0].legend(fontsize=7); axs[1].legend(fontsize=7)
    fig.suptitle(f"Retrospective active-learning simulation on the measured data (start {n0} random, {reps} repeats; mean ± SD)", fontsize=9)
    fig.tight_layout()
    save_fig(fig, "F30_active_learning_simulation", "experimental_design", "Active-learning simulation",
             "Model error and best value found vs number of experiments for random, exploration and EI strategies.", "— (project)",
             len(y_all), f"retrospective simulation, {reps} repeats")

## 20. Publication figure export
Every figure above is already saved as PNG (300 dpi), PDF and SVG (vector) in its topic folder. This step copies the
core set into `figures/publication/` with paper-style numbering and writes the **figure manifest** (CSV + Markdown).

In [ ]:
PUBLICATION_SET = [
    ("F01_scatter_matrix", "Fig01_scatter_matrix"), ("F06_correlation_heatmap", "Fig02_correlation"),
    ("F04_monte_carlo_outliers", "Fig03_outliers"), ("F09_grouped_benchmark", "Fig04_grouped_benchmark"),
    ("F12_taylor_diagrams", "Fig05_taylor"), ("F14_cross_plots", "Fig06_cross_plots"),
    ("F21_gp_parity_intervals", "Fig07_gp_parity_intervals"), ("F22_interval_calibration", "Fig08_calibration"),
    ("F24_response_surfaces", "Fig09_response_surfaces"), ("F25_response_curves", "Fig10_response_curves"),
    ("F18_rf_importance_and_shap", "Fig11_importance_shap"), ("F30_active_learning_simulation", "Fig12_active_learning"),
]
with step("20. Publication figure export"):
    copied = 0
    for key, pub in PUBLICATION_SET:
        for ext in FIG_FORMATS:
            src = next((FIG_DIRS[f] / f"{TARGET}_{key}.{ext}" for f in FIG_FOLDERS if (FIG_DIRS[f] / f"{TARGET}_{key}.{ext}").exists()), None)
            if src is None:
                continue
            shutil.copy2(src, FIG_DIRS["publication"] / f"{TARGET}_{pub}.{ext}"); copied += 1
    missing_pub = [k for k, _ in PUBLICATION_SET if not any(m["figure_id"] == f"{TARGET}_{k}" for m in MANIFEST)]
    if missing_pub:
        note("Publication figures not produced in this run: " + ", ".join(missing_pub))
    MAN = pd.DataFrame(MANIFEST)
    MAN.to_csv(DIRS["results"] / f"{TARGET}_figure_manifest.csv", index=False)
    md = [f"# Figure manifest - {TARGET} ({datetime.datetime.now():%Y-%m-%d %H:%M})", "",
          "| ID | Section | Title | Paper analogue | n | Validation | Files |", "|---|---|---|---|---|---|---|"]
    for _, r in MAN.iterrows():
        md.append(f"| {r.figure_id} | {r.section} | {r.title} | {r.paper_analogue} | {r.n} | {r.validation} | {r.files} |")
    (DIRS["figures"] / f"{TARGET}_figure_manifest.md").write_text("\n".join(md), encoding="utf-8")
    print(f"{len(MAN)} figures × {len(FIG_FORMATS)} formats saved; {copied} files copied to figures/publication/.")
    display(MAN[["figure_id", "folder", "title", "paper_analogue", "n", "validation"]])

## 21. Final results summary
Collects the key numbers (all computed in this run), the status of every section and the list of analyses that the
current data cannot support. Saved to `results/<TARGET>_run_summary.json`.

In [ ]:
with step("21. Final results summary"):
    S = {"run_finished": f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}", "runtime_min": round((time.time() - T_START) / 60, 1),
         "environment": "Google Colab" if IN_COLAB else "non-Colab", "gpu": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None,
         "target": TARGET, "target_description": TARGET_DESC, "unit": UNIT, "data_file": str(RAW_PATH), "data_sha256": DATA_HASH,
         "n_samples_modelled": int(len(y_all)), "split": SPLIT_DESC, "seed": SEED, "versions": VERSIONS}
    try:
        tm = METRICS_LONG[METRICS_LONG.model == BEST].set_index("set")
        S["best_model_by_validation"] = BEST
        S["best_model_metrics"] = {s: {k: round(float(tm.loc[s, k]), 4) for k in ("R2", "RMSE", "MRD")} for s in tm.index}
    except Exception:
        pass
    if "GMET" in globals():
        S["grouped_cv_top3"] = {s: g.sort_values("R2", ascending=False).head(3)[["model", "R2", "RMSE"]].round(3).to_dict("records")
                                for s, g in GMET.groupby("scheme")}
    if "GPSUM" in globals():
        S["gp_uncertainty"] = GPSUM.round(4).reset_index().to_dict("records"); S["k95"] = round(K95, 3)
    if "EXPLOIT" in globals():
        S["next_experiment_exploit_top3"] = EXPLOIT.head(3)[["IR", "Act_Temp", "Act_Time", "pred", "lo95", "hi95"]].round(1).to_dict("records")
        S["next_experiment_explore_top3"] = EXPLORE.head(3)[["IR", "Act_Temp", "Act_Time", "pred", "lo95", "hi95"]].round(1).to_dict("records")
    UNAVAILABLE = [
        "Experimental noise / repeatability: no replicate measurements exist, so measurement error cannot be estimated.",
        "Prospective validation: the suggested experiments (Section 19) have not been run yet; their outcomes are unknown.",
        "Extrapolation beyond 800-1200 °C, 0-90 min or IR 0-6, and hold-out of the edge levels (800/1200 °C, 0/90 min), "
        "are not evaluated.",
        "Yield modelling for IR ≠ 4 (one point each for IR 1, 2, 3, 5, 6; none for IR 0).",
        "Reference-paper analyses that do not transfer: Abraham descriptors, pore volume and logCe are not part of this dataset; "
        "the paper's model-schematic figures (Figs. 1-5) are diagrams, not data, and are not reproduced.",
    ]
    S["unavailable_analyses"] = UNAVAILABLE; S["notes"] = NOTES
    S["section_status"] = {k: v[0] for k, v in STATUS.items()}
    S["figures"] = len(MANIFEST)
    with open(DIRS["results"] / f"{TARGET}_run_summary.json", "w", encoding="utf-8") as fh:
        json.dump(S, fh, indent=2, default=str)

    display(Markdown(f"### Run summary - {TARGET_DESC}"))
    print(f"Samples modelled : {len(y_all)}   ({SPLIT_DESC})")
    if "best_model_by_validation" in S:
        b = S["best_model_metrics"]
        print(f"Best model (validation R²): {BEST}  ->  " + ", ".join(f"{s}: R² {v['R2']:.3f}, RMSE {v['RMSE']:.1f} {UNIT}" for s, v in b.items()))
    if "GMET" in globals():
        for s, g in GMET.groupby("scheme"):
            top = g.sort_values("R2", ascending=False).iloc[0]
            print(f"{SCHEME_NAME[s]:28s}: best {top.model} R² {top.R2:.3f}, RMSE {top.RMSE:.1f} {UNIT}")
    if "GPSUM" in globals():
        a = GPSUM.loc["All held-out levels"]
        print(f"GP 95 % interval coverage at unseen levels: raw {a.coverage95_raw:.0%}, calibrated (cross-fitted) "
              f"{a.coverage95_crossfit:.0%}; k95 = {K95:.2f}")
    if "EXPLOIT" in globals():
        r = EXPLOIT.iloc[0]
        print(f"Top suggested experiment (EI): " + ", ".join(f"{short(f)} {r[f]:g}" for f in ["IR", "Act_Temp", "Act_Time"]) +
              f" -> predicted {r.pred:.0f} {UNIT} (95 % {r.lo95:.0f}-{r.hi95:.0f})")
    print(f"Figures          : {len(MANIFEST)} (× {len(FIG_FORMATS)} formats) in {DIRS['figures']}")
    print(f"Runtime          : {S['runtime_min']} min")
    display(pd.DataFrame([(k, *v) for k, v in STATUS.items()], columns=["section", "status", "seconds", "error"]))
    display(Markdown("**Analyses not available with the current data**\n" + "\n".join(f"* {u}" for u in UNAVAILABLE)))
    if NOTES:
        display(Markdown("**Notes and caveats from this run**\n" + "\n".join(f"* {n_}" for n_ in NOTES)))
    failed = [k for k, v in STATUS.items() if v[0] != "OK"]
    print("\nALL SECTIONS COMPLETED SUCCESSFULLY" if not failed else f"\n{len(failed)} section(s) failed: {failed}")